# MOSAIC Null & Missing-Value Detector
**Detection-only.** Read-only against source data. Ambiguous findings require steward review. Clear single-option policy rules are auto-decided and attributed to `MOSAIC_POLICY_ENGINE`.

| Cell | What it does | MOSAIC §§ |
|---|---|---|
| 1 – Config | Parameters / widgets | — |
| 2 – Constants | Tags, allowlists, severity weights | §1–§8 |
| 3 – Discovery | Table & column inventory from information_schema | — |
| 4 – Profiler | One SQL pass per table, all columns, all counts | §1–§6 |
| 5 – AI classifier | Semantic column type (name / DOB / DOD / amount ...) | §2 §5 §6 |
| 6 – Rule engine | Deterministic rules; one function per rule group | §1–§6 |
| 7 – Write | Append findings to results Delta table | — |
| 8 – Summary | Blocking findings first, then ranked tables & columns | §8 |

> **Hard constraint:** UPDATE, MERGE, DELETE, CTAS on source tables are never executed.


# MOSAIC Null & Missing-Value Detector — Rule Reference

This notebook is **detection-only**. It never modifies source data. Every finding is a signal for a data steward to review — not an instruction to act.

---

## How to read a finding

Each finding row in the results table contains:

| Field | What it means |
|---|---|
| `classification` | The rule tag that fired (see below) |
| `rule_ref` | The MOSAIC standard section that defines the rule |
| `affected_count` / `affected_pct` | How many rows triggered the rule and what share of the column that is |
| `semantic_type` | What the notebook thinks the column *is* (name, DOB, amount, etc.) |
| `confidence` | How confident the classification is (`high`, `medium`, `low`, `heuristic`, `ai_degraded`) |
| `recommended_action` | Plain-English guidance — always requires steward review before action |
| `standardization_rule` | JSON array of options the steward can choose from. Each option has an `option_key`, a human-readable `label`, a concrete `sql` transform, and `notes` with constraints and caveats. The steward picks one option and records its `option_key` in `decided_action`. |
| `decided_action` | Auto-populated only for an undisputed semantic classification with exactly one policy option; otherwise NULL until a steward chooses an `option_key`. |
| `decided_by` | `MOSAIC_POLICY_ENGINE` for guarded auto-decisions; otherwise the steward identity after review. |
| `dictionary_strategy_hint` | Suggested data-dictionary strategy (A/B/C/D/null). See the **Strategy hint reference** section below for full definitions and which tags map to each. |
| `needs_steward_review` | `true` when the finding cannot be resolved without a domain decision |

**Decision workflow**

1. Run the notebook — unambiguous single-option rules are auto-decided; ambiguous findings retain `decided_action = NULL`.
2. Steward reviews each finding. For each one, read `standardization_rule` (a JSON array of options), pick the right `option_key`, and record it:
```sql
UPDATE <out_catalog>.<out_schema>.<out_table>
SET    decided_action = '<option_key>',
       decided_by     = '<steward name>'
WHERE  run_id = '<run_id>'
  AND  table_name = '<table>'
  AND  column_name = '<column>'
  AND  classification = '<tag>';
```
3. Engineer queries Section 5 of the summary (or runs the query below) to get their work queue — all findings where a decision has been recorded:
```sql
SELECT table_name, column_name, classification,
       decided_action, decided_by, standardization_rule
FROM   <out_catalog>.<out_schema>.<out_table>
WHERE  decided_action IS NOT NULL
ORDER BY table_name, column_name;
```
4. Engineer reads the `sql` field from the matching option in `standardization_rule` and applies the transform in the Silver/Gold pipeline.

**Blocking findings** (`DOD_SENTINEL_VIOLATION`, `FUTURE_DOB`, `DOD_FUTURE`, `DOD_BEFORE_DOB`, `DOB_SUSPECT_SENTINEL`) appear first in the summary and **block dataset certification** per MOSAIC §8.

---

## §1 — Null profile

**Tag:** `NULL_PROFILE`

**What it checks:** For every column that contains any kind of blank value, reports a full breakdown of *how* the value is blank — because the type of blank determines the correct data-dictionary strategy.

| Blank variant | Stored as | Caught by |
|---|---|---|
| True missing | SQL `NULL` | `IS NULL` |
| Empty string | `''` | `col = ''` |
| Whitespace-only | `'   '` | `col != '' AND TRIM(col) = ''` |
| Blank after trim | either of the above | `TRIM(col) = ''` |

**Why it matters:** An analyst counting `IS NULL` misses empty strings and whitespace-only values entirely. The steward needs the full breakdown to pick the right data-dictionary strategy — see the **Strategy hint reference** section below.

**Action:** Assign a strategy in the data dictionary. No automatic conversion.

---

## §2 — String token rules

### `MAP_TO_NULL_CANDIDATE`

**What it checks:** After `TRIM()`, counts occurrences of tokens that should become SQL `NULL` in Silver/Gold:
- Empty string `''` (including whitespace-only values after trim)
- The literal string `NULL` / `null` / `Null` stored as text (common when ETL tools export unquoted NULLs)

**Why it matters:** These tokens pass `IS NOT NULL` checks silently and break aggregations and joins that expect true `NULL` semantics.

**Action:** Confirm with steward that no downstream system depends on the literal value, then convert to SQL `NULL` in the Silver transform. Strategy A hint.

---

### `PRESERVE_NAME_FIELD`

**What it checks:** The literal token `Null` / `NULL` in a column classified as a **person name**.

**Why it matters:** "Null" is a real surname. Converting it to SQL `NULL` corrupts identity matching and patient records. This was the specific example raised by Ryan Williams in the 2026-06-22 Architecture Sync.

**Action:** Never coerce to SQL `NULL` in a name column. Steward must explicitly confirm whether a specific value is a placeholder before any action.

---

### `PRESERVE_SENTINEL`

**What it checks:** Detects the following tokens (case-insensitive, after trim) in any string column:

| Token(s) | Meaning |
|---|---|
| `Unknown`, `Unk`, `UNK` | Value was not captured — user identified the absence |
| `N/A`, `NA`, `n/a`, `N.A.`, `Not Applicable` | The question does not apply to this record |
| `None` | Clinically or operationally distinct from NULL in many source systems |

**Why it matters:** These are **Strategy B curated sentinels** — they carry meaning about *why* data is absent. `N/A` ≠ `NULL`. Converting them silently destroys that context and is a policy violation. The Navina dataset example (Ryan Williams) is the canonical MOSAIC case.

**Action:** Do not convert to `NULL` without explicit steward sign-off and a data dictionary version bump. Consider normalising variants (e.g. `Unk` → `Unknown`) within the sentinel class.

---

### `BUSINESS_CODE_REVIEW`

**What it checks:** In columns with ≤ 20 distinct values, flags single or double character codes such as `N`, `U`, `X`, `?`, `-`, `Z`, `NP`, `NK` that are ambiguous between "not applicable" and a real business value.

**Why it matters:** `N` in a Gender column may mean "non-binary" (a real value) or "not collected" (a missing-value marker). The notebook cannot distinguish these from data alone. Auto-converting either way would corrupt the column.

**Action:** Confirm the code's meaning with the domain steward. Document it in the data dictionary before any standardisation.

---

### `THIRD_STATE_IN_FLAG`

**What it checks:** A column whose distinct values look like a boolean flag (Y/N, true/false, 1/0, T/F) but also contains a **third distinct value** beyond the expected pair.

**Examples of third states that trigger this rule:**
- `Y` / `N` / `U` — is `U` "unknown" or a data-entry error?
- `1` / `0` / `9` — is `9` a system default for "not set"?
- `Y` / `N` / `?` — informal "unclear"

**Why it matters:** Per MOSAIC §5, boolean/flag columns must use `NULL` for unknown — not a third string state in the same column. A third state makes the column semantically ambiguous and breaks BI logic.

**Action:** Steward must decide whether the third value is unknown (Strategy A / SQL NULL), not applicable (Strategy B / N/A), or a genuine documented business state. No strategy is inferred automatically.

---

## §3 / §5 — Date and numeric sentinel rules

### `KNOWN_NULL_SENTINEL`

**Tag fires for:** `1875-05-20`

**What it checks:** The Cobalt system uses `1875-05-20` as its internal default for "no date entered". This value appears wherever a Cobalt-sourced date field was never populated.

**Why it matters:** It is never a real calendar date. Leaving it in Silver/Gold contaminates date arithmetic, age calculations, and any range filter that uses this column.

**Action:** Convert to `NULL` after steward confirmation. If the column is Strategy C, quarantine instead. Register the sentinel in the data dictionary.

---

### `SUSPECT_SENTINEL`

**Tag fires for:** `1900-01-01`

**What it checks:** `1900-01-01` is a common default in spreadsheet serial-date encoding, SQL Server epoch arithmetic, and legacy ETL fill logic.

**Why it matters:** In some sources it may rarely be a genuine date (e.g. a historical record). The rule fires because the value is *suspect*, not because we know for certain it is a placeholder.

**Important:** This rule **never recommends auto-conversion** to `NULL`. The notebook deliberately uses the `SUSPECT_SENTINEL` tag (not `KNOWN_NULL_SENTINEL`) to force a profiling decision.

**Action:** Profile the source system to confirm intent. If confirmed as a placeholder, register in the dictionary and convert in Silver. If a DOB column fires this rule, the tag escalates to `DOB_SUSPECT_SENTINEL` (see §6).

---

### `MAX_DATE_SENTINEL`

**Tag fires for:** `9999-12-31`

**What it checks:** The classic "no end date" / "open for ever" placeholder. Common in effective-date ranges, subscription records, and coverage periods.

**Why it matters:** `9999-12-31` overflows many downstream tools and distorts date arithmetic (e.g. DATEDIFF returns implausible values). MOSAIC standard specifies `2199-01-01` as the approved open-ended sentinel for curated layers.

**Action depends on column type:**
- **Open-ended end-date columns** (`effective_to`, `valid_to`, `end_date`): replace with `2199-01-01`
- **Point-in-time columns**: convert to `NULL`
- **DOD columns**: `POLICY_VIOLATION` — see §6

Requires steward confirmation in all cases.

---

### `INVALID_DATE`

**What it checks:** Dates that cannot represent a real calendar date:
- Year `0000` (zero-year dates)
- Values stored as a date type that parse to `NULL` via `TRY_CAST` (meaning the stored value is not a valid date)

**Why it matters:** These values will produce `NULL` or errors in any downstream date calculation silently.

**Action:** Investigate source system behaviour. Route to quarantine. Never persist in curated layers.

---

### `CANDIDATE_MAGIC_DATE`

**What it checks:** Any single date value that appears in more than `magic_pct`% (default 1%) of non-null rows in a date column — excluding the known sentinel dates above.

**Why it matters:** A real date should not dominate a date column's distribution. When one date appears disproportionately, it is usually a system default, a batch-load artefact, or an undocumented sentinel that profiling missed.

**Examples seen in practice:** `2000-01-01` (millennium fill), `2023-01-01` (migration cutover date used as "unknown"), `1970-01-01` (Unix epoch zero).

**Action:** Investigate the source system and load history. If confirmed as a placeholder, register and convert. Confidence is `medium` — the notebook cannot determine intent from frequency alone.

---

### `ZERO_AS_MISSING_REVIEW`

**What it checks:** Zero values in columns the classifier identified as **measure or amount** columns (e.g. `charge_amount`, `fee`, `revenue`, `balance`).

**Why it matters:** In measure columns, stored `0` may mean "data was not collected" rather than a genuine zero value. This is invisible to standard null checks and silently distorts aggregations (sum, average, percentile).

**Note:** This rule does **not** fire for identifier, flag, or count columns where zero is a normal valid value.

**Action:** Confirm with domain steward whether `0` is a valid business value or a missing-data marker. If the latter, convert to `NULL`. Confidence is `medium`.

---

### `CANDIDATE_MAGIC_NUMBER`

**What it checks:** The following numeric values when they appear in more than `magic_pct`% of total rows:

| Value | Typical origin |
|---|---|
| `9999` | "Unknown" sentinel in legacy operational systems |
| `-1` | "Not recorded" or "not applicable" in many EHR exports |
| `99999` | Overflow or "not set" in fixed-width file formats |
| `-99`, `-9999` | Common coded missing values in claims and clinical data |
| `999` | Short-field equivalent of 9999 |

**Why it matters:** These values pass numeric type checks and appear in aggregations, producing misleading results (a mean that includes `9999` as a real value is meaningless).

**Action:** Steward investigation required. If confirmed as a sentinel, register in the dictionary and convert to `NULL` in Silver.

---

## §6 — Date of birth (DOB) rules

DOB is a high-impact PHI field used in EMPI identity matching, age/eligibility calculations, and cohort definitions. It gets its own rule group beyond the generic date rules above.

### `FUTURE_DOB`

**What it checks:** Any `date_of_birth` value greater than today.

**Why it matters:** A future birth date is clinically impossible and indicates a data-entry error or system default. Any row with a future DOB cannot be used in age calculations, eligibility windows, or cohort filters without investigation.

**Action:** Quarantine all rows with future DOB. Do not silently null or drop — steward review required. Blocks certification.

---

### `PLAUSIBILITY_REVIEW`

**What it checks:** DOB values that imply an age outside `[0, age_max]` years (default `age_max = 120`, configurable via widget).

- Age < 0: DOB is in the future (caught separately by `FUTURE_DOB`)
- Age > 120: biologically implausible; likely a placeholder or data-entry error

**Why it matters:** Implausible ages corrupt age-based metrics, eligibility windows, and EMPI matching scores. A wrong DOB imputed at scale propagates errors into every age-derived calculation.

**Action:** Flag for plausibility review. Do not silently drop or null. Steward must confirm before any action.

---

### `DOB_SUSPECT_SENTINEL`

**What it checks:** `1900-01-01` specifically in a DOB column.

**Why it matters:** Unlike a generic date column where `1900-01-01` is *suspect* (may rarely be genuine), in a DOB column no real patient has this date. It is unambiguously a placeholder — but per MOSAIC policy it is still an **unregistered sentinel** and must not be auto-nulled. It could have been loaded intentionally by the source system with a specific meaning.

**Action:** Quarantine candidate — do **not** auto-null. Steward must investigate the source system, confirm intent, register the sentinel in the dictionary, and then convert. Severity escalated vs. generic `SUSPECT_SENTINEL`.

---

### `DOB_MISSING`

**What it checks:** Rows where the DOB column is `NULL`.

**Why it matters:** Per MOSAIC §6, DOB is a **Strategy C field** for member/patient master records — zero nulls tolerated in certified products unless a documented waiver exists. A missing DOB means the row cannot be used in age calculations or EMPI matching.

**Action:** Rows with missing DOB should be quarantined rather than defaulted. Never impute DOB — a wrong DOB corrupts EMPI matching and every age-based metric downstream.

---

## §6 — Date of death (DOD) rules

DOD is used in mortality reporting, survivorship analysis, and CMS/mortality-completeness reconciliation.

### `DOD_SENTINEL_VIOLATION` ⚠ BLOCKS CERTIFICATION

**What it checks:** Any future sentinel value in a DOD column: `9999-12-31` or `2199-01-01`.

**Why it matters:** Per MOSAIC §6, a living patient must have `NULL` in the DOD field — never a sentinel date. A stored `9999-12-31` or `2199-01-01` falsely implies the patient has a death date on record, which corrupts mortality logic, CMS reporting, and survivorship queries.

This is the most severe finding in the null/missing ruleset. It directly contradicts MOSAIC policy and **blocks certification** of any dataset that contains it.

**Action:** Immediate steward and governance review required. Convert to `NULL`. Do not allow into certified Gold layer.

---

### `DOD_FUTURE`

**What it checks:** Any DOD value greater than today.

**Why it matters:** A death cannot occur in the future. A future DOD is either a data-entry error or a system default that was incorrectly loaded into this column.

**Action:** Quarantine all rows with future DOD. Steward investigation required. Blocks certification.

---

### `DOD_BEFORE_DOB`

**What it checks:** Rows where `DOD < DOB` (cross-column check — requires a `date_of_birth` column to be present and classified in the same table).

**Why it matters:** Chronologically impossible. A person cannot die before they are born. These rows indicate a data-entry error, a misloaded field, or a column mapping error.

**Action:** Quarantine all violating rows. Do not silently null either field. Steward and source-system investigation required. Blocks certification.

---

---

## Strategy hint reference

Every finding carries a `dictionary_strategy_hint` value that tells the steward which data-dictionary strategy best fits the pattern detected. The table below defines each strategy, what it means in practice, and which tags typically produce it.

| Hint | Strategy name | What it means in practice | Typically seen on |
|---|---|---|---|
| **A** | Nullable / absent | SQL `NULL` is the approved representation for an absent or unknown value. Blank strings and literal `NULL` tokens normalize here, except literal Null/NULL in name fields. | `MAP_TO_NULL_CANDIDATE`; nullable numeric/date `NULL_PROFILE` |
| **B** | Curated explicit sentinel | Preserve a documented meaningful sentinel such as `Unknown`, `N/A`, or `None`; canonicalize variants without erasing their meaning. Open-ended range boundaries use `2199-01-01`. | `PRESERVE_SENTINEL`; open-ended range boundaries |
| **C** | Required / quarantine on absence | The field is mandatory at the certified grain. Missing or invalid values are quarantined; never silently default or impute. | `DOB_MISSING` for patient/member master records |
| **D** | Not consumer-ready without transformation | Raw/source semantics cannot be exposed directly in Gold. A documented transformation must resolve the value before certification. Never infer this strategy from a short code alone. | Steward-confirmed source-specific cases only |
| *(null)* | No safe hint | Meaning cannot be determined from values or null rate alone. The steward must establish semantics before choosing A-D. | `BUSINESS_CODE_REVIEW`, `THIRD_STATE_IN_FLAG`, `SUSPECT_SENTINEL`, generic string `NULL_PROFILE` |

> The hint is a **starting point**, not a decision. Steward sign-off is required before any strategy is recorded in the data dictionary or any transform is deployed.

## Extensibility — adding new rule modules

This notebook is designed to grow. The null/missing ruleset is **Module 001**. Future modules (format standardisation, referential integrity, coding system validation, etc.) follow the same pattern:

1. Add tag(s) to `TAGS` in **Cell 2** with the MOSAIC section reference as a comment
2. Add severity weights to `SEVERITY` in **Cell 2**
3. Write a new `_check_<topic>(tbl, col, stats, sem, ...) -> list` function in **Cell 6**
4. Call it in the main loop at the bottom of **Cell 6** alongside the existing checks
5. That is all — no changes to any other cell


The findings schema, write cell, and summary cell are rule-agnostic and require no modification.


In [0]:
import re, json, uuid, hashlib
from datetime import datetime, date
from typing import Any, Dict, List, Optional, Tuple
from collections import defaultdict
from pyspark.sql import functions as F, DataFrame

# Read a Databricks widget or fall back to a default.
def _w(name: str, default) -> str:
    try:
        dbutils.widgets.text(name, str(default))
        return dbutils.widgets.get(name)
    except Exception:
        return str(default)


def _qident(value: str) -> str:
    """Quote one Unity Catalog identifier component safely."""
    value = str(value)
    if not value.strip():
        raise ValueError("Catalog, schema, table, and column names cannot be blank.")
    return "`" + value.replace("`", "``") + "`"


def _fq(*parts: str) -> str:
    return ".".join(_qident(p) for p in parts)


def _sql_lit(value: Any) -> str:
    """Quote a SQL string literal safely."""
    return "'" + str(value).replace("'", "''") + "'"


def _int_widget(name: str, default: int, minimum: int = 0) -> int:
    value = int(_w(name, default))
    if value < minimum:
        raise ValueError(f"{name} must be >= {minimum}; received {value}.")
    return value


def _float_widget(name: str, default: float, minimum: float = 0.0) -> float:
    value = float(_w(name, default))
    if value < minimum:
        raise ValueError(f"{name} must be >= {minimum}; received {value}.")
    return value

CFG: Dict[str, Any] = {
    # Source
    "catalog":      _w("catalog",      "data_classification_source"),
    "schema":       _w("schema",       "null_missing_value_detector"),
    "table_filter": _w("table_filter", ""),        # regex or blank = all
    "skip_views":   _w("skip_views",   "true").strip().lower() == "true",

    # Sampling  (value-level analysis only -- aggregates always use full table)
    "sample_size":  _int_widget("sample_size", 10_000, 1),
    "seed":         _int_widget("seed", 42, 0),
    # token_limit (default 200) — §2 string token rules only.
    # After sampling, the scanner collects the top-N most-frequent distinct trimmed values
    # per string column and checks them against the null/sentinel/code allowlists.
    # Capped at 200 to avoid scanning thousands of unique free-text values: in a column
    # like "notes" or "comments" most distinct values are meaningless for sentinel detection
    # and the rule is only useful on low-to-medium cardinality columns anyway.
    # Raise if you have columns with many legitimate code values you want to inspect fully.
    # Lower on very wide schemas to keep the sample pass fast.
    "token_limit":  _int_widget("token_limit", 200, 1),

    # Results
    "out_catalog":  _w("out_catalog",  "data_classification_target"),
    "out_schema":   _w("out_schema",   "data_classification_output"),
    "out_table":    _w("out_table",    "null_missing_findings"),

    # AI
    "ai_model":     _w("ai_model",     "databricks-meta-llama-3-3-70b-instruct"),
    "enable_ai":    _w("enable_ai",    "true").strip().lower() == "true",
    # Raw source values are never sent for semantic classification. Optional token AI
    # is off by default and restricted to low-cardinality non-PHI code columns.
    "enable_ai_token_discovery": _w("enable_ai_token_discovery", "false").strip().lower() == "true",

    # Thresholds
    "age_max":      _int_widget("age_max", 120, 1),
    # magic_pct (default 1.0) — controls the frequency threshold for two rule checks:
    #   CANDIDATE_MAGIC_DATE:   a specific date value is flagged if it appears in >= magic_pct%
    #     of non-null rows in a date column. Example: 2000-01-01 appearing in 3,000 of 200,000
    #     rows (1.5%) is suspicious enough to flag as a possible system default or load artefact.
    #   CANDIDATE_MAGIC_NUMBER: same logic for numeric sentinels like 9999, -1, 99999 --
    #     flagged if they appear in >= magic_pct% of total rows in a numeric column.
    # Raise (e.g. 5.0) for fewer, higher-confidence flags on noisy sources.
    # Lower (e.g. 0.1) to catch rarer sentinels in very large tables.
    "magic_pct":    _float_widget("magic_pct", 1.0, 0.0),
}

for _name in ("catalog", "schema", "out_catalog", "out_schema", "out_table"):
    if not str(CFG[_name]).strip():
        raise ValueError(f"{_name} cannot be blank.")
try:
    re.compile(CFG["table_filter"] or ".*", re.I)
except re.error as exc:
    raise ValueError(f"Invalid table_filter regular expression: {exc}") from exc
if CFG["magic_pct"] > 100:
    raise ValueError("magic_pct must be between 0 and 100.")

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.utcnow().isoformat()
TODAY  = date.today()

print(f"Run      : {RUN_ID}")
print(f"Scope    : {CFG['catalog']}.{CFG['schema']}")
print(f"AI       : {'on -> ' + CFG['ai_model'] if CFG['enable_ai'] else 'off (heuristic-only)'}")
print(f"Results  : {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")


In [0]:
# ---------------------------------------------------------------------------
# HOW TO ADD A NEW RULE MODULE (e.g. format standardisation, ref integrity)
#   1. Add tag(s) to TAGS below with a comment showing the rule ref
#   2. Add the tag to SEVERITY with a weight
#   3. Add detection logic in Cell 6 as a new _check_<topic>() function
#      and call it in the main loop at the bottom of that cell
# ---------------------------------------------------------------------------

# Classification tags: tag_name -> MOSAIC standard section reference
TAGS = {
    # §1 Null profile
    "NULL_PROFILE":             "§1",

    # §2 String token rules
    "MAP_TO_NULL_CANDIDATE":    "§2",
    "PRESERVE_NAME_FIELD":      "§2,§5",
    "PRESERVE_SENTINEL":        "§2",
    "BUSINESS_CODE_REVIEW":     "§2",

    # §3/§5 Date sentinels
    "KNOWN_NULL_SENTINEL":      "§3,§5",
    "SUSPECT_SENTINEL":         "§3",
    "UNREGISTERED_MISSING_MARKER": "§2,§3",
    "MAX_DATE_SENTINEL":        "§3",
    "INVALID_DATE":             "§3",
    "CANDIDATE_MAGIC_DATE":     "§3",

    # §3/§5 Numeric sentinels
    "ZERO_AS_MISSING_REVIEW":   "§5",
    "CANDIDATE_MAGIC_NUMBER":   "§3,§5",

    # §5 Boolean
    "THIRD_STATE_IN_FLAG":      "§5",

    # §6 DOB / DOD
    "FUTURE_DOB":               "§6-DOB",
    "PLAUSIBILITY_REVIEW":      "§6-DOB",
    "DOB_SUSPECT_SENTINEL":     "§6-DOB",
    "DOB_MISSING":              "§6-DOB",
    "DOD_FUTURE":               "§6-DOD",
    "DOD_SENTINEL_VIOLATION":   "§6,§8",   # BLOCKS certification
    "DOD_BEFORE_DOB":           "§6-DOD",

}

# Severity weights for impact scoring in the summary (Cell 9)
SEVERITY: Dict[str, int] = {
    "DOD_SENTINEL_VIOLATION":   10,   # blocks certification
    "FUTURE_DOB":                9,
    "DOD_FUTURE":                9,
    "DOD_BEFORE_DOB":            9,
    "DOB_MISSING":               7,
    "PLAUSIBILITY_REVIEW":       7,
    "KNOWN_NULL_SENTINEL":       6,
    "MAP_TO_NULL_CANDIDATE":     5,
    "THIRD_STATE_IN_FLAG":       5,
    "INVALID_DATE":              5,
    "MAX_DATE_SENTINEL":         4,
    "SUSPECT_SENTINEL":          4,
    "UNREGISTERED_MISSING_MARKER": 4,
    "DOB_SUSPECT_SENTINEL":      4,
    "CANDIDATE_MAGIC_DATE":      3,
    "CANDIDATE_MAGIC_NUMBER":    3,
    "ZERO_AS_MISSING_REVIEW":    3,
    "PRESERVE_SENTINEL":         2,
    "PRESERVE_NAME_FIELD":       2,
    "BUSINESS_CODE_REVIEW":      2,
    "NULL_PROFILE":              1,
}

# §2 Token allowlists (evaluated after TRIM() and lower())
NULL_TOKENS     = {"", "null"}          # recommend SQL NULL (with name-field exception)
SENTINEL_TOKENS = {"unknown", "unk", "n/a", "na", "n.a.", "not applicable", "none"}  # preserve
AMBIG_CODES     = {"N", "U", "X", "?", "-", "Z", "NP", "NK"}                        # low-card review
BOOL_SETS: List[set] = [
    {"y", "n"}, {"yes", "no"}, {"true", "false"}, {"1", "0"}, {"t", "f"}
]

# §3 Pinned sentinel dates
DATE_COBALT = date(1875,  5, 20)   # Cobalt empty-date -> KNOWN_NULL_SENTINEL
DATE_EPOCH  = date(1900,  1,  1)   # DB default       -> SUSPECT_SENTINEL
DATE_MAX    = date(9999, 12, 31)   # No-end marker    -> MAX_DATE_SENTINEL
DATE_MOSAIC = date(2199,  1,  1)   # MOSAIC approved open-ended sentinel
KNOWN_DATES = {DATE_COBALT, DATE_EPOCH, DATE_MAX, DATE_MOSAIC}

# §3/§5 Numeric magic values
MAGIC_NUMBERS = {9999, -1, 99999, -99, -9999, 999}

# Semantic types used for column classification
SEMANTIC_TYPES = {
    "person_name", "date_of_birth", "date_of_death", "open_ended_end_date",
    "identifier", "measure_amount", "business_code", "free_text", "other"
}

# Token-boundary semantic patterns. Column names are normalized to snake_case
# before matching, preventing false positives such as count in country and charge in discharge.
RE_NAME = re.compile(
    r"(?:^|_)(?:(?:first|last|full|middle|sur|given|family|maiden|preferred|clinician|provider|physician|doctor)_)?name(?:_|$)",
    re.I,
)
RE_DOB = re.compile(r"(?:^|_)(?:dob|date_of_birth|birth_date|birthdate)(?:_|$)", re.I)
RE_DOD = re.compile(r"(?:^|_)(?:dod|date_of_death|death_date|deathdate)(?:_|$)", re.I)
RE_ENDDATE_EXCLUDE = re.compile(
    r"^(?:transaction|event|created|updated|modified|recorded|admission|discharge|"
    r"order|payment|invoice|processed|submitted|posted|issued|started|opened)_date(?:_|$)",
    re.I,
)
RE_ENDDATE = re.compile(
    r"(?:^|_)(?:effective_to|valid_to|end_date|to_date|expiry|expiration|termination_date|coverage_end|close_date)(?:_|$)",
    re.I,
)
RE_MEASURE = re.compile(
    r"(?:^|_)(?:amount|amt|value|val|price|cost|revenue|qty|quantity|count|total|sum|balance|charge|fee)(?:_|$)",
    re.I,
)
RE_ID = re.compile(r"(?:^id(?:_|$)|(?:^|_)(?:id|key|number|num|no|nhs|mrn|chi)(?:_|$))", re.I)
RE_BUSINESS_CODE = re.compile(
    r"(?:^|_)(?:code|status|type|category|gender|sex|flag|method|unit|grade|class|reason)(?:_|$)",
    re.I,
)

print("Constants loaded")
print(f"  {len(TAGS)} tags  |  {len(SEVERITY)} severity weights  |  {len(SENTINEL_TOKENS)} preserved sentinels")

# ---------------------------------------------------------------------------
# STANDARDIZATION RULES
#
# What engineers should implement in the Silver/Gold transform for each
# classification. These are detection findings only -- no data is changed
# by this notebook. The transform instructions below describe what SHOULD
# happen in the pipeline once a steward has confirmed the finding.
#
# Rules are written as concrete transform steps, not general guidance.
# Every rule references the MOSAIC standard section it comes from.
# ---------------------------------------------------------------------------

# Each entry is a list of options the steward can choose from.
# option_key  : short identifier stored in decided_action when chosen
# label       : plain-English name shown to the steward
# sql         : concrete Silver/Gold transform the engineer applies
# notes       : constraints, caveats, or prerequisite steps
STANDARDIZATION_RULES: Dict[str, list] = {

    # ── §1 / §2 / §5  String and null normalisation ──────────────────────────

    # NULL_PROFILE options are type-aware and built by the rule engine at finding time.
    # See NULL_PROFILE_OPTIONS below -- the rule engine selects the right option set
    # based on the column's dtype (string / numeric / date / other).
    # "NULL_PROFILE" is intentionally absent from this dict.

    "MAP_TO_NULL_CANDIDATE": [
        {"option_key": "convert_to_null",
         "label":      "Convert empty / whitespace / literal-NULL to SQL NULL",
         "sql":        "CASE WHEN TRIM(col) = '' THEN NULL WHEN LOWER(TRIM(col)) = 'null' THEN NULL ELSE TRIM(col) END",
         "notes":      "Standard §2 rule. Do NOT apply to name columns -- see PRESERVE_NAME_FIELD. "
                       "Confirm no downstream system depends on the literal value before deploying."},
        {"option_key": "preserve_as_is",
         "label":      "Preserve as-is (downstream contract requires literal value)",
         "sql":        "col",
         "notes":      "Only if a certified BI contract requires the literal '' or 'NULL' string. "
                       "Must be documented as an exception in the data dictionary (§5)."},
    ],

    "PRESERVE_NAME_FIELD": [
        {"option_key": "trim_only",
         "label":      "Apply TRIM() only -- preserve literal Null/NULL as a surname",
         "sql":        "TRIM(col)",
         "notes":      "'Null' is a legitimate surname (§2, §5). Never coerce to SQL NULL. "
                       "Empty strings and whitespace-only values in the same column still follow "
                       "the standard MAP_TO_NULL rule (TRIM -> '' -> NULL)."},
        {"option_key": "trim_and_null_blanks",
         "label":      "Trim and convert blank variants to NULL, preserve literal Null surname",
         "sql":        "CASE WHEN TRIM(col) = '' THEN NULL ELSE TRIM(col) END",
         "notes":      "Converts empty/whitespace to NULL but keeps the string 'Null'/'NULL' intact. "
                       "Use when both blank variants and the surname value coexist in the column."},
    ],

    "PRESERVE_SENTINEL": [
        {"option_key": "normalise_canonical",
         "label":      "Normalise variants to canonical sentinel strings",
         "sql":        "CASE WHEN LOWER(TRIM(col)) IN ('unknown','unk') THEN 'Unknown' "
                       "     WHEN LOWER(TRIM(col)) IN ('n/a','na','n.a.','not applicable') THEN 'N/A' "
                       "     WHEN LOWER(TRIM(col)) = 'none' THEN 'None' ELSE TRIM(col) END",
         "notes":      "Preserve as Strategy B sentinel. Do NOT convert to NULL. "
                       "Expose as a distinct string value in Gold (§2)."},
        {"option_key": "convert_to_null_with_signoff",
         "label":      "Convert to NULL (requires per-field steward sign-off)",
         "sql":        "CASE WHEN LOWER(TRIM(col)) IN ('unknown','unk','n/a','na','n.a.','not applicable','none') "
                       "     THEN NULL ELSE col END",
         "notes":      "Only permitted with documented per-field steward sign-off and "
                       "a data-dictionary version bump (§2, §10). Do not apply without approval."},
    ],

    "BUSINESS_CODE_REVIEW": [
        {"option_key": "preserve_business_code",
         "label":      "Preserve as a valid business code",
         "sql":        "col",
         "notes":      "Add the code and its definition to the data dictionary. No transform needed."},
        {"option_key": "convert_to_null",
         "label":      "Convert to NULL (confirmed as missing-value marker)",
         "sql":        "-- Generated per finding with the safely quoted detected code.",
         "notes":      "Only if steward confirms the code has no business meaning and represents "
                       "missing data. The rule engine inserts the detected code safely. Document in PR (§10)."},
        {"option_key": "map_to_sentinel",
         "label":      "Map to N/A or Unknown sentinel",
         "sql":        "-- Generated per finding with the safely quoted detected code.",
         "notes":      "Use when the code means 'not applicable' or 'not collected'. "
                       "The rule engine inserts the detected code safely. Document decision (§10)."},
    ],

    "THIRD_STATE_IN_FLAG": [
        {"option_key": "map_third_to_null",
         "label":      "Map third value to NULL (it means unknown)",
         "sql":        "-- Generated per finding with the safely quoted detected third-state token.",
         "notes":      "Use when the third state means 'unknown'. The rule engine inserts the detected token safely. "
                       "Boolean columns must use NULL for unknown, not a string state (§5)."},
        {"option_key": "promote_to_code_column",
         "label":      "Promote column to a string code column",
         "sql":        "-- Change column type to STRING; add all values including third state to data dictionary.",
         "notes":      "Use when the third value is a real business code, not a missing marker. "
                       "Update data dictionary and downstream BI contracts."},
        {"option_key": "map_to_na_sentinel",
         "label":      "Map third value to N/A sentinel (it means not applicable)",
         "sql":        "-- Generated per finding with the safely quoted detected third-state token.",
         "notes":      "Use when the third state means 'not applicable'. Preserve as a Strategy B curated sentinel and document it (§5)."},
    ],

    # ── §3 / §5  Date sentinel normalisation ─────────────────────────────────

    "KNOWN_NULL_SENTINEL": [
        {"option_key": "convert_to_null",
         "label":      "Convert 1875-05-20 to NULL",
         "sql":        "CASE WHEN col = DATE'1875-05-20' THEN NULL ELSE col END",
         "notes":      "1875-05-20 is the Cobalt system empty-date -- never a real date (§3). "
                       "Register in the data dictionary before deploying (§10)."},
        {"option_key": "quarantine_strategy_c",
         "label":      "Quarantine row instead of nulling (Strategy C column)",
         "sql":        "-- Route rows WHERE col = DATE'1875-05-20' to quarantine table; exclude from Silver grain.",
         "notes":      "Use when the column is Strategy C (required). "
                       "Do not null -- quarantine and log for steward review."},
    ],

    "SUSPECT_SENTINEL": [
        {"option_key": "convert_to_null_confirmed",
         "label":      "Convert to NULL (confirmed as placeholder after profiling)",
         "sql":        "CASE WHEN col = DATE'1900-01-01' THEN NULL ELSE col END",
         "notes":      "Only apply after steward profiling confirms the value is a system default "
                       "in this specific source. Never auto-convert (§3). Register in dictionary."},
        {"option_key": "preserve_pending_review",
         "label":      "Preserve as-is pending steward profiling",
         "sql":        "col",
         "notes":      "Use when profiling is not yet complete. Flag for monitoring. "
                       "1900-01-01 may rarely be a genuine date in historical records."},
    ],

    "MAX_DATE_SENTINEL": [
        {"option_key": "replace_with_mosaic_sentinel",
         "label":      "Replace 9999-12-31 with 2199-01-01 (open-ended range)",
         "sql":        "CASE WHEN col = DATE'9999-12-31' THEN DATE'2199-01-01' ELSE col END",
         "notes":      "Use for open-ended end-date columns (effective_to, valid_to). "
                       "2199-01-01 is the MOSAIC approved sentinel for open-ended ranges (§3)."},
        {"option_key": "convert_to_null",
         "label":      "Convert 9999-12-31 to NULL (point-in-time unknown)",
         "sql":        "CASE WHEN col = DATE'9999-12-31' THEN NULL ELSE col END",
         "notes":      "Use for point-in-time date columns where 9999-12-31 means 'unknown'. "
                       "9999-12-31 must never persist in Gold (§3)."},
    ],

    "INVALID_DATE": [
        {"option_key": "null_and_quarantine",
         "label":      "Convert to NULL and route to quarantine log",
         "sql":        "CASE WHEN TRY_CAST(col AS DATE) IS NULL OR YEAR(TRY_CAST(col AS DATE)) = 0 "
                       "     THEN NULL ELSE TRY_CAST(col AS DATE) END",
         "notes":      "Route affected rows to a quarantine log for source investigation (§5). "
                       "Zero-dates and unparseable values must not persist in curated layers."},
    ],

    "CANDIDATE_MAGIC_DATE": [
        {"option_key": "convert_to_null_confirmed",
         "label":      "Convert confirmed sentinel date to NULL",
         "sql":        "CASE WHEN col = DATE'<confirmed_sentinel>' THEN NULL ELSE col END",
         "notes":      "Only after steward confirms the date is a system default. "
                       "Replace <confirmed_sentinel> with the actual date. Register in dictionary (§10)."},
        {"option_key": "replace_with_mosaic_sentinel",
         "label":      "Replace confirmed sentinel with 2199-01-01 (open-ended range column)",
         "sql":        "CASE WHEN col = DATE'<confirmed_sentinel>' THEN DATE'2199-01-01' ELSE col END",
         "notes":      "Use only for open-ended end-date columns. Replace <confirmed_sentinel>."},
        {"option_key": "preserve_pending_review",
         "label":      "Preserve as-is pending steward confirmation",
         "sql":        "col",
         "notes":      "Use when profiling is not yet complete. Flag for ongoing monitoring."},
    ],

    # ── §3 / §5  Numeric sentinel normalisation ───────────────────────────────

    "ZERO_AS_MISSING_REVIEW": [
        {"option_key": "convert_to_null_confirmed",
         "label":      "Convert 0 to NULL (confirmed as missing-data marker)",
         "sql":        "CASE WHEN col = 0 THEN NULL ELSE col END",
         "notes":      "Only after steward confirms 0 represents missing data, not a genuine zero. "
                       "Do not use silent COALESCE (§4). Document decision in data dictionary."},
        {"option_key": "preserve_valid_zero",
         "label":      "Preserve 0 as a valid business value",
         "sql":        "col",
         "notes":      "Use when 0 is a genuine business value (e.g. a fee waiver, a zero balance). "
                       "Document in data dictionary that 0 is intentional."},
    ],

    "CANDIDATE_MAGIC_NUMBER": [
        {"option_key": "convert_to_null_confirmed",
         "label":      "Convert confirmed sentinel value to NULL",
         "sql":        "CASE WHEN col = <confirmed_value> THEN NULL ELSE col END",
         "notes":      "Only after steward confirms the value is a sentinel, not a real measurement. "
                       "Replace <confirmed_value>. Register in dictionary; reference in pipeline PR (§10)."},
        {"option_key": "preserve_pending_review",
         "label":      "Preserve as-is pending steward confirmation",
         "sql":        "col",
         "notes":      "Use when the value has not yet been confirmed as a sentinel. "
                       "Flag for monitoring."},
    ],

    # ── §6  DOB standardisation ───────────────────────────────────────────────

    "FUTURE_DOB": [
        {"option_key": "quarantine",
         "label":      "Quarantine row for steward investigation",
         "sql":        "-- Route rows WHERE dob > CURRENT_DATE to quarantine table; exclude from Silver grain.",
         "notes":      "Do not null silently. Never impute a DOB (§6-DOB). "
                       "Exclude from EMPI matching and age calculations until resolved."},
    ],

    "PLAUSIBILITY_REVIEW": [
        {"option_key": "quarantine",
         "label":      "Quarantine implausible DOB rows for steward review",
         "sql":        "-- Route rows WHERE derived_age < 0 OR derived_age > age_max to quarantine table.",
         "notes":      "Do not null or drop. Preserve source value for investigation (§6-DOB). "
                       "Log quarantine count for monthly steward review."},
    ],

    "DOB_SUSPECT_SENTINEL": [
        {"option_key": "quarantine_pending_confirmation",
         "label":      "Quarantine pending steward confirmation of sentinel meaning",
         "sql":        "-- Route rows WHERE dob = DATE'1900-01-01' to quarantine table.",
         "notes":      "Do NOT auto-null. 1900-01-01 is unregistered in this source (§6-DOB). "
                       "After steward confirms and registers: use convert_to_null option."},
        {"option_key": "convert_to_null_confirmed",
         "label":      "Convert to NULL (confirmed as placeholder, registered in dictionary)",
         "sql":        "CASE WHEN dob = DATE'1900-01-01' THEN NULL ELSE dob END",
         "notes":      "Only after steward investigation and dictionary registration. "
                       "Never use 1900-01-01 as a valid DOB in EMPI or eligibility calculations."},
    ],

    "DOB_MISSING": [
        {"option_key": "quarantine",
         "label":      "Quarantine rows with missing DOB (Strategy C)",
         "sql":        "-- Route rows WHERE dob IS NULL to quarantine table; exclude from Silver grain.",
         "notes":      "DOB is Strategy C for patient master records -- zero nulls tolerated (§6-DOB). "
                       "Never impute. Document any waiver in the data product definition."},
    ],

    # ── §6  DOD standardisation ───────────────────────────────────────────────

    "DOD_SENTINEL_VIOLATION": [
        {"option_key": "convert_to_null",
         "label":      "Convert sentinel to NULL -- IMMEDIATE ACTION REQUIRED",
         "sql":        "CASE WHEN dod IN (DATE'9999-12-31', DATE'2199-01-01') THEN NULL ELSE dod END",
         "notes":      "BLOCKS certification (§6-DOD, §8). Living patients must have NULL DOD. "
                       "Deploy before certification. Document remediation in change log (§9, §10)."},
    ],

    "DOD_FUTURE": [
        {"option_key": "quarantine",
         "label":      "Quarantine rows with future DOD",
         "sql":        "-- Route rows WHERE dod > CURRENT_DATE to quarantine table.",
         "notes":      "A death cannot occur in the future (§6-DOD). Preserve source value. "
                       "Exclude from mortality reporting and survivorship until resolved."},
    ],

    "DOD_BEFORE_DOB": [
        {"option_key": "quarantine",
         "label":      "Quarantine rows where DOD precedes DOB",
         "sql":        "-- Route rows WHERE dod < dob to quarantine table.",
         "notes":      "Chronologically impossible (§6-DOD). Do not alter either field. "
                       "Exclude from mortality, survivorship, and EMPI matching until resolved."},
    ],

}

# Type-aware options for NULL_PROFILE findings.
# The rule engine calls NULL_PROFILE_OPTIONS[type_family] to populate
# standardization_rule based on the column's actual data type.
NULL_PROFILE_OPTIONS: Dict[str, list] = {

    "string": [
        {"option_key": "convert_trim_null",
         "label":      "Trim then convert blank to NULL (Strategy A default)",
         "sql":        "CASE WHEN TRIM(col) = '' THEN NULL ELSE TRIM(col) END",
         "notes":      "Apply TRIM() first; whitespace-only becomes '' which then maps to NULL. "
                       "Preferred default per §2. Confirm no downstream BI contract requires ''."},
        {"option_key": "preserve_empty",
         "label":      "Preserve empty string as-is (BI contract exception)",
         "sql":        "TRIM(col)",
         "notes":      "Only if a certified BI contract explicitly requires '' over NULL. "
                       "Must be documented as an exception in the data dictionary (§5)."},
        {"option_key": "assign_strategy_first",
         "label":      "Assign data-dictionary strategy before deciding transform",
         "sql":        "-- No transform until strategy A/B/C/D is assigned in the data dictionary.",
         "notes":      "Use when the blank pattern has multiple variants (null + empty + whitespace) "
                       "that may need different handling. Steward must assign strategy first (§1)."},
    ],

    "numeric": [
        {"option_key": "accept_null_as_missing",
         "label":      "Accept SQL NULL as the missing-value representation (Strategy A)",
         "sql":        "-- No transform needed. NULL already represents missing for this numeric column.",
         "notes":      "Document the null rate and expected threshold in the data dictionary (§1, §5). "
                       "Do not substitute 0 or any other value for NULL (§5)."},
        {"option_key": "assign_strategy_first",
         "label":      "Assign data-dictionary strategy before deciding action",
         "sql":        "-- No transform until strategy A/B/C/D is assigned in the data dictionary.",
         "notes":      "Use when the business meaning of NULL is unclear. Steward must confirm "
                       "whether NULL means not-collected, unknown, or not-applicable (§1)."},
    ],

    "date": [
        {"option_key": "accept_null_as_missing",
         "label":      "Accept SQL NULL as the missing-value representation (Strategy A)",
         "sql":        "-- No transform needed. NULL already represents missing for this date column.",
         "notes":      "Document in the data dictionary (§1, §5). "
                       "Verify no sentinel dates are hiding as NULL -- run the date sentinel checks."},
        {"option_key": "assign_strategy_first",
         "label":      "Assign data-dictionary strategy before deciding action",
         "sql":        "-- No transform until strategy A/B/C/D is assigned in the data dictionary.",
         "notes":      "For DOB columns, Strategy C (quarantine) applies -- see DOB_MISSING rule."},
    ],

    "other": [
        {"option_key": "assign_strategy_first",
         "label":      "Assign data-dictionary strategy before deciding action",
         "sql":        "-- No transform until strategy A/B/C/D is assigned in the data dictionary.",
         "notes":      "Determine the column type and business meaning of NULL before proceeding (§1)."},
    ],
}

print(f"Standardization rules loaded: {len(STANDARDIZATION_RULES)} tag(s), " f"{sum(len(v) for v in STANDARDIZATION_RULES.values())} total options")


In [0]:
# ---------------------------------------------------------------------------
# Reads Unity Catalog information_schema -- never touches source data.
# Builds three objects used by all later cells:
#   tables     : List[str]
#   table_cols : Dict[str, List[(col, dtype)]]
# ---------------------------------------------------------------------------

cat, sch = CFG["catalog"], CFG["schema"]

# 1. Tables. Identifier components and metadata predicates are quoted independently.
view_clause = "AND table_type NOT IN ('VIEW', 'MATERIALIZED_VIEW')" if CFG["skip_views"] else ""
tables: List[str] = [
    r.table_name
    for r in spark.sql(f"""
        SELECT table_name
        FROM   {_qident(cat)}.information_schema.tables
        WHERE  table_schema = {_sql_lit(sch)} {view_clause}
        ORDER BY table_name
    """).collect()
]

if CFG["table_filter"].strip():
    pat = re.compile(CFG["table_filter"], re.I)
    tables = [t for t in tables if pat.search(t)]

# Never recursively scan the findings table when output is inside the source scope.
if cat == CFG["out_catalog"] and sch == CFG["out_schema"]:
    tables = [t for t in tables if t != CFG["out_table"]]

if not tables:
    raise ValueError(f"No tables found in {cat}.{sch} -- check catalog/schema/table_filter.")

# 2. Columns. Avoid a generated IN list so schemas with many tables and unusual
# table names remain safe and do not exceed SQL statement-length limits.
selected_tables = set(tables)
table_cols: Dict[str, List[Tuple[str, str]]] = defaultdict(list)
for r in spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM   {_qident(cat)}.information_schema.columns
    WHERE  table_schema = {_sql_lit(sch)}
    ORDER BY table_name, ordinal_position
""").collect():
    if r.table_name in selected_tables:
        table_cols[r.table_name].append((r.column_name, r.data_type.upper()))

total_cols = sum(len(v) for v in table_cols.values())
print(f"Scope   : {cat}.{sch}")
print(f"Tables  : {len(tables)}")
print(f"Columns : {total_cols}")


In [0]:
# ---------------------------------------------------------------------------
# KEY DESIGN: one SQL query per table -- not one per column.
#
# The previous version called get_full_agg() once per column, so a 100-column
# table triggered 100 SQL round trips. Here we build one SELECT with
# aggregation expressions for every column in the table, collect once,
# and store per-column stats in a plain dict.
#
# Outputs:
#   table_stats   : Dict[str, Dict[str, dict]]  -- table -> col -> stat dict
#   table_samples : Dict[str, DataFrame]         -- reproducible sample per table
# ---------------------------------------------------------------------------

def _is_str(dt: str) -> bool:
    return any(dt.startswith(p) for p in ("STRING","VARCHAR","CHAR","TEXT"))

def _is_num(dt: str) -> bool:
    return dt in {"INT","INTEGER","BIGINT","LONG","DOUBLE","FLOAT","SMALLINT","TINYINT"} \
        or any(dt.startswith(p) for p in ("DECIMAL","NUMERIC","FLOAT","DOUBLE"))

def _is_dt(dt: str) -> bool:
    return dt in {"DATE","TIMESTAMP","TIMESTAMP_NTZ"} or dt.startswith("TIMESTAMP")


def profile_table(tbl: str, cols: List[Tuple[str, str]]) -> Tuple[int, Dict[str, dict]]:
    # One exact aggregate pass per table. Position-based aliases prevent collisions
    # between unusual column names such as `a b` and `a_b`.
    exprs = ["COUNT(*) AS __total__"]
    aliases = {}
    for idx, (col, dtype) in enumerate(cols):
        c = _qident(col)
        key = f"c{idx}"
        aliases[col] = key
        exprs.append(f"COUNT(*) - COUNT({c}) AS {_qident('__null__' + key)}")
        if _is_str(dtype):
            exprs.extend([
                f"COUNT(CASE WHEN {c} = '' THEN 1 END) AS {_qident('__empty__' + key)}",
                f"COUNT(CASE WHEN {c} != '' AND TRIM({c}) = '' THEN 1 END) AS {_qident('__ws__' + key)}",
                f"COUNT(CASE WHEN {c} IS NOT NULL AND TRIM({c}) = '' THEN 1 END) AS {_qident('__btrim__' + key)}",
                f"COUNT(CASE WHEN LOWER(TRIM({c})) = 'null' THEN 1 END) AS {_qident('__literal_null__' + key)}",
                f"COUNT(CASE WHEN LOWER(TRIM({c})) IN ('unknown','unk') THEN 1 END) AS {_qident('__sent_unknown__' + key)}",
                f"COUNT(CASE WHEN LOWER(TRIM({c})) IN ('n/a','na','n.a.','not applicable') THEN 1 END) AS {_qident('__sent_na__' + key)}",
                f"COUNT(CASE WHEN LOWER(TRIM({c})) = 'none' THEN 1 END) AS {_qident('__sent_none__' + key)}",
                f"APPROX_COUNT_DISTINCT(TRIM({c})) AS {_qident('__approx_distinct__' + key)}",
            ])

    expr_str = f"SELECT {', '.join(exprs)} FROM {_fq(cat, sch, tbl)}"
    row = spark.sql(expr_str).collect()[0].asDict()
    total = row["__total__"]
    stats = {}
    for col, dtype in cols:
        key = aliases[col]
        stats[col] = {
            "total": total,
            "null": row.get(f"__null__{key}", 0) or 0,
            "empty": row.get(f"__empty__{key}", 0) or 0,
            "ws": row.get(f"__ws__{key}", 0) or 0,
            "btrim": row.get(f"__btrim__{key}", 0) or 0,
            "literal_null": row.get(f"__literal_null__{key}", 0) or 0,
            "sent_unknown": row.get(f"__sent_unknown__{key}", 0) or 0,
            "sent_na": row.get(f"__sent_na__{key}", 0) or 0,
            "sent_none": row.get(f"__sent_none__{key}", 0) or 0,
            "approx_distinct": row.get(f"__approx_distinct__{key}", 0) or 0,
            "dtype": dtype,
        }
    return total, stats


def get_sample(tbl: str) -> DataFrame:
    # TABLESAMPLE (n ROWS) uses block-level sampling -- no full-scan shuffle.
    # Falls back to .sample() if not supported.
    fq = _fq(cat, sch, tbl)
    n  = CFG["sample_size"]
    try:
        return spark.sql(f"SELECT * FROM {fq} TABLESAMPLE ({n} ROWS)")
    except Exception:
        total = spark.sql(f"SELECT COUNT(*) AS n FROM {fq}").collect()[0]["n"]
        return (
            spark.table(fq)
            .sample(withReplacement=False, fraction=min(1.0, n / max(1, total)), seed=CFG["seed"])
            .limit(n)
        )


def mask(values: list, n: int = 5) -> list:
    return [str(v) for v in values if v is not None][:n]


# Run profiler
table_stats:   Dict[str, Dict[str, dict]] = {}
table_samples: Dict[str, DataFrame]       = {}

for tbl in tables:
    cols = table_cols[tbl]
    total, stats = profile_table(tbl, cols)
    if total == 0:
        print(f"[SKIP] {tbl} -- empty table")
        continue
    table_stats[tbl]   = stats
    table_samples[tbl] = get_sample(tbl)
    print(f"  ok {tbl}: {total:,} rows / {len(cols)} cols")

print(f"\nProfiler done. {len(table_stats)} table(s) ready.")


In [0]:
# ---------------------------------------------------------------------------
# Determines what each column IS semantically:
#   person_name, date_of_birth, date_of_death, open_ended_end_date,
#   identifier, measure_amount, business_code, free_text, other
#
# This drives which rules fire in Cell 6:
#   - name-field: literal NULL is PRESERVE, not MAP_TO_NULL
#   - date_of_birth / date_of_death: §6 plausibility rules activate
#   - measure_amount: zero-as-missing check activates
#   - open_ended_end_date: 9999-12-31 -> recommend 2199-01-01
#
# DESIGN: one ai_query() call per TABLE with all columns batched.
# Heuristic regex always runs as a baseline; AI disagreement lowers
# confidence and sets needs_review = True.
#
# Output: sem_map[table][col] = {type, source, confidence, needs_review}
# ---------------------------------------------------------------------------

def _normalise_column_name(col: str) -> str:
    # Handles snake_case, spaces, punctuation, and camelCase consistently.
    value = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", str(col))
    return re.sub(r"[^A-Za-z0-9]+", "_", value).strip("_").lower()


def _heuristic(col: str, dtype: str) -> str:
    name = _normalise_column_name(col)
    if RE_DOB.search(name): return "date_of_birth"
    if RE_DOD.search(name): return "date_of_death"
    if RE_NAME.search(name): return "person_name"
    if RE_ENDDATE.search(name) and not RE_ENDDATE_EXCLUDE.search(name):
        return "open_ended_end_date"
    # Amount/measure semantics require a numeric physical type. A STRING column
    # named value_text must never become measure_amount from its name alone.
    if _is_num(dtype) and RE_MEASURE.search(name): return "measure_amount"
    if RE_ID.search(name): return "identifier"
    if RE_BUSINESS_CODE.search(name): return "business_code"
    return "other"


def _ai_query(prompt: str) -> str:
    raw = spark.sql(f"SELECT ai_query({_sql_lit(CFG['ai_model'])}, {_sql_lit(prompt)}, failOnError => false) AS r").collect()[0]["r"]
    if hasattr(raw, "errorStatus") and raw.errorStatus:
        raise ValueError(raw.errorStatus)
    return raw.result if hasattr(raw, "result") else str(raw)


def classify_table_columns(tbl: str, sample_df: DataFrame) -> Dict[str, dict]:
    cols = table_cols[tbl]
    heuristics = {col: _heuristic(col, dtype) for col, dtype in cols}

    # Enterprise privacy boundary: semantic classification receives metadata only.
    # Raw source values (including possible PHI/PII) are never sent to ai_query.
    col_signals = {}
    for col, dtype in cols:
        s = table_stats.get(tbl, {}).get(col, {})
        total = s.get("total", 1) or 1
        col_signals[col] = {
            "null_pct": round(s.get("null", 0) / total * 100, 1),
            "approx_distinct": int(s.get("approx_distinct", 0) or 0),
        }

    if not CFG["enable_ai"]:
        return {
            col: {"type": heuristics[col], "source": "heuristic", "confidence": "heuristic", "needs_review": False}
            for col, _ in cols
        }

    # One prompt for all columns in this table.
    # Include cardinality signals so the AI can distinguish codes (low distinct) from free-text (high distinct).
    payload = json.dumps(
        [{
            "col":            col,
            "dtype":          dtype,
            "null_pct":       col_signals.get(col, {}).get("null_pct", 0),
            "approx_distinct": col_signals.get(col, {}).get("approx_distinct", 0),
        } for col, dtype in cols],
        default=str
    )
    prompt = (
        "Strict data classifier. Reply ONLY with a JSON array -- no prose, no markdown. "
        f"Table: {tbl}. Use the table name as strong context when classifying columns. "
        "Classify each column semantic type as exactly one of: "
        "person_name, date_of_birth, date_of_death, open_ended_end_date, "
        "identifier, measure_amount, business_code, free_text, other. "
        "Use these definitions: "
        "person_name=a human name field (first, last, full, any language/abbreviation); "
        "date_of_birth=birth date (any language: dt_nasci, fecha_nac, dob, birthdate); "
        "date_of_death=death date (any language: dt_obit, fecha_def, dod, deathdate); "
        "open_ended_end_date=a range END boundary only (effective_to, valid_to, expiry, coverage_end, termination_date) -- NOT event dates; "
        "identifier=a unique key, code, or reference number; "
        "measure_amount=a numeric quantity, amount, price, or measurement; "
        "business_code=a low-cardinality coded field (status, type, category, gender, flag); "
        "free_text=unstructured or high-cardinality text; "
        "other=none of the above. "
        "IMPORTANT: point-in-time event dates (admission, discharge, transaction, order, payment, created, updated) "
        "are NOT open_ended_end_date -- use other. "
        "Low approx_distinct (<20) usually means business_code, not free_text. "
        f"Columns: {payload}. "
        'Return: [{"col":"<name>","type":"<type>","confidence":"high|medium|low"}]'
    )

    try:
        ai_map = {item["col"]: item for item in json.loads(_ai_query(prompt)) if "col" in item}
        result = {}
        for col, dtype in cols:
            ai      = ai_map.get(col, {})
            ai_type = ai.get("type", "other") if ai.get("type") in SEMANTIC_TYPES else "other"
            ai_conf = ai.get("confidence", "low")
            h_type  = heuristics[col]

            # Enforce physical/semantic compatibility before AI output can route rules.
            if ai_type == "measure_amount" and not _is_num(dtype):
                ai_type = "free_text" if _is_str(dtype) else "other"
                ai_conf = "medium"
            if ai_type in ("person_name", "business_code", "free_text") and not _is_str(dtype):
                ai_type = "other"
                ai_conf = "medium"

            # Hard exclusion: RE_ENDDATE_EXCLUDE is a definitive rule, not a soft signal.
            # If the heuristic excluded a column from open_ended_end_date (e.g. discharge_date,
            # admission_date), the AI cannot override that exclusion -- these are point-in-time
            # event dates regardless of what the AI infers from samples.
            if ai_type == "open_ended_end_date" and RE_ENDDATE_EXCLUDE.search(col):
                ai_type = "other"
                ai_conf = "medium"

            disagree = h_type != "other" and h_type != ai_type
            # Preservation-priority heuristic matches (name/DOB/DOD/range-end) win over
            # AI disagreement. AI may enrich otherwise-unclassified columns, but cannot
            # silently downgrade a high-impact deterministic semantic match.
            chosen_type = h_type if h_type != "other" else ai_type
            result[col] = {
                "type": chosen_type,
                "source": "both" if h_type == ai_type and h_type != "other" else "heuristic" if h_type != "other" else "ai",
                "confidence": "high" if h_type != "other" and not disagree else "medium" if h_type != "other" else ai_conf,
                "needs_review": disagree or ai_conf == "low",
            }
        return result
    except Exception as e:
        print(f"  [WARN] AI classify failed for {tbl}: {e} -- using heuristics")
        return {
            col: {"type": heuristics[col], "source": "heuristic", "confidence": "ai_degraded", "needs_review": True}
            for col, _ in cols
        }


sem_map: Dict[str, Dict[str, dict]] = {}
for tbl, sample_df in table_samples.items():
    sem_map[tbl] = classify_table_columns(tbl, sample_df)
    print(f"  ok {tbl}")

print(f"\nClassification done. {len(sem_map)} table(s).")


In [0]:
# ---------------------------------------------------------------------------
# HOW TO READ THIS CELL
#
# Each MOSAIC rule group is one standalone function:
#   _check_null_profile()    §1
#   _check_string_tokens()   §2
#   _check_date_sentinels()  §3/§5/§6
#   _check_numeric()         §3/§5
#
# Every function receives the column context and returns List[dict].
# The main loop at the bottom calls them in sequence.
#
# HOW TO ADD A NEW RULE MODULE:
#   1. Write a new function: def _check_<topic>(tbl, col, stats, sem, ...) -> list
#   2. Add one line in the main loop: new += _check_<topic>(...)
#   3. Add your tag(s) to TAGS and SEVERITY in Cell 2 -- that's it.
# ---------------------------------------------------------------------------

from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, BooleanType


def _finding(tbl, col, dtype, sem, tag, count, total, samples, action,
             hint=None, confidence="high",
             std_options=None, auto_decided_action=None) -> dict:
    pct = round(count / total * 100, 4) if total else 0.0
    options = std_options if std_options is not None else STANDARDIZATION_RULES.get(tag, [])
    option_keys = {o.get("option_key") for o in options}
    can_auto_decide = (
        auto_decided_action is not None
        and len(options) == 1
        and auto_decided_action in option_keys
        and not bool(sem.get("needs_review", False))
    )
    return {
        "run_id": RUN_ID, "run_ts": RUN_TS,
        "catalog": cat, "schema": sch,
        "table_name": tbl, "column_name": col, "data_type": dtype,
        "semantic_type":            sem.get("type",   "other"),
        "semantic_source":          sem.get("source", "heuristic"),
        "rule_ref":                 TAGS.get(tag, "§?"),
        "classification":           tag,
        "affected_count":           int(count),
        "affected_pct":             float(pct),
        "total_rows":               int(total),
        "sample_values":            json.dumps(samples, default=str),
        "recommended_action":       action,
        "dictionary_strategy_hint": hint,
        "confidence":               confidence,
        # needs_steward_review computed below based on whether decided_action was set
        # std_options overrides the lookup when the rule engine provides type-aware options (e.g. NULL_PROFILE).
        "standardization_rule":     json.dumps(
            options,
            ensure_ascii=False
        ),
        # Auto-decisions are limited to an explicit engine recommendation whose
        # standardization_rule has exactly one valid option and whose semantic type is not disputed.
        "decided_action":           auto_decided_action if can_auto_decide else None,
        "needs_steward_review":     not can_auto_decide,
        "decided_by":               "MOSAIC_POLICY_ENGINE" if can_auto_decide else None,
    }


# ── §1 Null profile ──────────────────────────────────────────────────────────
def _null_profile_type_family(dtype: str) -> str:
    if _is_str(dtype): return "string"
    if _is_num(dtype): return "numeric"
    if _is_dt(dtype):  return "date"
    return "other"


def _check_null_profile(tbl, col, stats, sem) -> list:
    s        = stats[col]
    blanks   = s["null"] + s["empty"] + s["ws"]
    sem_type = sem.get("type", "other")
    findings = []

    # Fix #2: DOD NULL_PROFILE suppressed entirely.
    # NULL DOD = living patient (§6). A high null rate is correct by design,
    # not a data quality issue. DOD rules handle all DOD-specific checks.
    if sem_type == "date_of_death":
        return []

    # Fix #3: DOB_MISSING runs here from pre-computed stats so it fires
    # independently of whether the date SQL query in _check_date_sentinels succeeds.
    if sem_type == "date_of_birth" and s["null"] > 0:
        findings.append(_finding(tbl, col, s["dtype"], sem, "DOB_MISSING",
            s["null"], s["total"], [],
            "Missing DOB violates Strategy C completeness for patient master records (§6-DOB). "
            "Quarantine rows with NULL DOB; never default or impute.",
            hint="C", confidence="high",
            auto_decided_action="quarantine",
        ))

    if not blanks:
        return findings

    dtype       = s["dtype"]
    type_family = _null_profile_type_family(dtype)

    # For numeric and date columns the procedure is unambiguous (§5):
    # NULL is the correct missing-value representation. Pre-populate decided_action.
    # Exception: open_ended_end_date -- NULL means "no end / still active", which
    # is the correct business state, not a missing-data condition.
    # String columns always require a steward strategy decision (A/B/C/D).
    is_open_ended = (sem_type == "open_ended_end_date")

    auto_decide = {
        "numeric": "accept_null_as_missing",
        "date":    "accept_null_as_missing" if not is_open_ended else None,
    }
    decided = auto_decide.get(type_family)

    options = NULL_PROFILE_OPTIONS.get(type_family, NULL_PROFILE_OPTIONS["other"])
    if decided:
        options = [o for o in options if o["option_key"] == decided]
    elif is_open_ended and type_family == "date":
        sentinel_literal = "DATE'2199-01-01'" if dtype == "DATE" else "TIMESTAMP'2199-01-01 00:00:00'"
        options = [{
            "option_key": "replace_null_with_mosaic_sentinel",
            "label": "Replace NULL with the approved 2199-01-01 open-ended sentinel",
            "sql": f"CASE WHEN col IS NULL THEN {sentinel_literal} ELSE col END",
            "notes": "MOSAIC §3 requires 2199-01-01 for no-known-end/far-future range boundaries. "
                     "Document the field as Strategy B and do not use this rule for DOD.",
        }]
        decided = "replace_null_with_mosaic_sentinel"

    action_map = {
        "string":  "Blank values detected (null / empty / whitespace). Assign Strategy A/B/C/D and confirm trim-then-null transform (§1, §2).",
        "numeric": "NULL values in a numeric column. Per §5, NULL is the correct missing-value representation -- document null rate in the data dictionary.",
        "date":    (
            "NULL values in an open-ended range boundary must be standardised to the approved 2199-01-01 sentinel in curated layers. "
            "Verify that DOD and point-in-time dates are not classified as open-ended."
        ) if is_open_ended else
            "NULL values in a date column. Per §5, NULL is the correct missing-value representation -- verify no sentinel dates are hidden.",
        "other":   "NULL values detected. Assign data-dictionary strategy before deciding on a transform (§1).",
    }

    findings.append(_finding(tbl, col, dtype, sem, "NULL_PROFILE",
        s["null"], s["total"],
        [{"null": s["null"], "empty": s["empty"], "whitespace_only": s["ws"], "blank_after_trim": s["btrim"]}],
        action_map[type_family],
        hint="B" if is_open_ended else "A" if type_family in ("numeric", "date") else None,
        confidence="high",
        std_options=options,
        auto_decided_action=decided,
    ))
    return findings


# ── §2 String token rules ────────────────────────────────────────────────────
def _check_string_tokens(tbl, col, stats, sem, sample_df, dist_vals_in=None) -> list:
    s        = stats[col]
    dtype    = s["dtype"]
    total    = s["total"]
    sem_type = sem.get("type", "other")

    if not _is_str(dtype):
        return []

    # Use pre-computed dist_vals if provided (avoids a second Spark query when
    # the caller already collected tokens for the AI check).
    if dist_vals_in is not None:
        dist_vals = dist_vals_in
    else:
        dist = (
            sample_df
            .select(F.trim(F.col(_qident(col))).alias("tv"))
            .filter(F.col("tv").isNotNull())
            .groupBy("tv").count()
            .orderBy(F.col("count").desc())
            .limit(CFG["token_limit"])
            .collect()
        )
        dist_vals = [(r["tv"], r["count"]) for r in dist]
    # Exact aggregate counts below must still fire when a rare token was absent
    # from the bounded discovery sample.
    n_distinct = int(s.get("approx_distinct", len(dist_vals)) or 0)
    findings   = []

    # §2.1 Exact full-table NULL-mappable token counts. The sample is used only
    # for discovery; affected_count and affected_pct always come from the aggregate profile.
    literal_null_count = int(s.get("literal_null", 0) or 0)
    blank_token_count = int(s.get("empty", 0) or 0) + int(s.get("ws", 0) or 0)
    if sem_type == "person_name" and literal_null_count:
        findings.append(_finding(tbl, col, dtype, sem, "PRESERVE_NAME_FIELD",
            literal_null_count, total, ["Null/NULL"],
            "Literal Null/NULL in a name column is preservation-priority and must not be coerced to SQL NULL.",
            confidence=sem.get("confidence", "medium"),
        ))

    null_count = blank_token_count + (0 if sem_type == "person_name" else literal_null_count)
    if null_count:
        null_samples = (["<empty-or-whitespace>"] if blank_token_count else []) + (["NULL"] if literal_null_count and sem_type != "person_name" else [])
        name_safe_options = [{
            "option_key": "trim_and_null_blanks",
            "label": "Trim and convert blanks to NULL while preserving literal Null/NULL names",
            "sql": "CASE WHEN TRIM(col) = '' THEN NULL ELSE TRIM(col) END",
            "notes": "Name-field precedence rule: never apply the generic literal-NULL conversion to this column.",
        }] if sem_type == "person_name" else None
        findings.append(_finding(tbl, col, dtype, sem, "MAP_TO_NULL_CANDIDATE",
            null_count, total, null_samples,
            "Empty/whitespace/literal-NULL tokens must map to SQL NULL in curated layers, subject to the name-field exception.",
            hint="A", std_options=name_safe_options,
        ))

    # §2.2 Exact counts grouped by canonical Strategy B sentinel.
    sentinel_groups = [
        ("Unknown", int(s.get("sent_unknown", 0) or 0), ["unknown", "unk"]),
        ("N/A", int(s.get("sent_na", 0) or 0), ["n/a", "na", "n.a.", "not applicable"]),
        ("None", int(s.get("sent_none", 0) or 0), ["none"]),
    ]
    for canonical, cnt, variants in sentinel_groups:
        if cnt:
            findings.append(_finding(tbl, col, dtype, sem, "PRESERVE_SENTINEL",
                cnt, total, [canonical],
                f"{cnt} value(s) map to canonical '{canonical}'. Preserve this business sentinel; do not convert it to NULL without per-field sign-off.",
                hint="B",
            ))

    # §5 Boolean third state -- evaluated first so we know which tokens it covers.
    # Any token claimed by THIRD_STATE_IN_FLAG is suppressed from BUSINESS_CODE_REVIEW
    # to avoid the steward seeing two overlapping findings for the same value.
    third_state_tokens: set = set()
    boolean_base_tokens: set = set()
    if n_distinct <= 3:
        lower_set = {tv.strip().lower() for tv, _ in dist_vals}
        for bset in BOOL_SETS:
            extra = lower_set - bset - {""}
            # A third state exists only when the complete true/false base pair is present.
            if len(extra) == 1 and bset.issubset(lower_set):
                boolean_base_tokens = set(bset)
                ex = next(iter(extra))
                actual_token, cnt = next(
                    ((tv, c) for tv, c in dist_vals if tv.strip().lower() == ex),
                    (ex, 0),
                )
                third_state_tokens.add(ex)
                token_sql = _sql_lit(actual_token)
                third_options = [
                    {"option_key": "map_third_to_null", "label": "Map third value to NULL (it means unknown)",
                     "sql": f"CASE WHEN TRIM(col) = {token_sql} THEN NULL ELSE TRIM(col) END",
                     "notes": f"Applies specifically to detected token {actual_token!r}. Boolean unknown must use SQL NULL (§5)."},
                    {"option_key": "promote_to_code_column", "label": "Preserve as a documented business state",
                     "sql": "TRIM(col)",
                     "notes": f"Use only when detected token {actual_token!r} is a genuine business state; update the dictionary and BI contracts."},
                    {"option_key": "map_to_na_sentinel", "label": "Map third value to N/A (it means not applicable)",
                     "sql": f"CASE WHEN TRIM(col) = {token_sql} THEN 'N/A' ELSE TRIM(col) END",
                     "notes": f"Applies specifically to detected token {actual_token!r}; preserve as a Strategy B curated sentinel (§5)."},
                ]
                findings.append(_finding(tbl, col, dtype, sem, "THIRD_STATE_IN_FLAG",
                    cnt, total, mask([actual_token]),
                    f"Flag column has a third value {actual_token!r}. Determine whether it means unknown (Strategy A/SQL NULL), "
                    "not applicable (Strategy B/N/A), or a genuine documented business state.",
                    hint=None, std_options=third_options,
                ))
                break

    # §2.3 Ambiguous business codes (low-cardinality columns only).
    # Skip tokens already handled by THIRD_STATE_IN_FLAG -- they are more specifically
    # characterised there and the steward only needs one finding per value.
    if n_distinct <= 20 and sem_type not in ("person_name",):
        for tv, cnt in dist_vals:
            if tv.strip().upper() in AMBIG_CODES:
                if tv.strip().lower() not in third_state_tokens and tv.strip().lower() not in boolean_base_tokens:
                    token_sql = _sql_lit(tv)
                    business_options = [
                        {"option_key": "preserve_business_code", "label": "Preserve as a valid business code",
                         "sql": "TRIM(col)",
                         "notes": f"Preserves detected token {tv!r}; add its definition to the data dictionary."},
                        {"option_key": "convert_to_null", "label": "Convert confirmed missing marker to SQL NULL",
                         "sql": f"CASE WHEN TRIM(col) = {token_sql} THEN NULL ELSE TRIM(col) END",
                         "notes": f"Applies specifically to detected token {tv!r}; requires steward confirmation and dictionary change (§10)."},
                        {"option_key": "map_to_sentinel", "label": "Map confirmed code to N/A",
                         "sql": f"CASE WHEN TRIM(col) = {token_sql} THEN 'N/A' ELSE TRIM(col) END",
                         "notes": f"Use only if detected token {tv!r} means not applicable; document the Strategy B mapping (§10)."},
                    ]
                    findings.append(_finding(tbl, col, dtype, sem, "BUSINESS_CODE_REVIEW",
                        cnt, total, mask([tv]),
                        f"Code {tv!r} is ambiguous in this low-cardinality column. "
                        "Confirm its domain meaning before assigning any null strategy or transform.",
                        hint=None, confidence="medium", std_options=business_options,
                    ))

    return findings


# ── §3/§5/§6 Date sentinel rules ─────────────────────────────────────────────
def _check_date_sentinels(tbl, col, stats, sem, dob_col: Optional[str]) -> list:
    s        = stats[col]
    dtype    = s["dtype"]
    total    = s["total"]
    null_cnt = s["null"]
    sem_type = sem.get("type", "other")

    needs_date_rules = _is_dt(dtype) or sem_type in (
        "date_of_birth", "date_of_death", "open_ended_end_date"
    )
    if not needs_date_rules:
        return []

    fq = f"`{cat}`.`{sch}`.`{tbl}`"
    dc = f"`{col}`" if _is_dt(dtype) else f"TRY_CAST(`{col}` AS DATE)"

    try:
        row = spark.sql(f"""
            SELECT
                COUNTIF(TRY_CAST({dc} AS DATE) = DATE'1875-05-20')       AS c_cobalt,
                COUNTIF(TRY_CAST({dc} AS DATE) = DATE'1900-01-01')       AS c_epoch,
                COUNTIF(TRY_CAST({dc} AS DATE) = DATE'9999-12-31')       AS c_max,
                COUNTIF(TRY_CAST({dc} AS DATE) = DATE'2199-01-01')       AS c_mosaic,
                COUNTIF(YEAR(TRY_CAST({dc} AS DATE)) = 0
                        OR (`{col}` IS NOT NULL
                            AND TRY_CAST({dc} AS DATE) IS NULL))          AS c_invalid,
                COUNTIF(TRY_CAST({dc} AS DATE) > DATE'{TODAY}')          AS c_future
            FROM {fq}
        """).collect()[0].asDict()
    except Exception as e:
        print(f"  [WARN] Date query failed for {tbl}.{col}: {e}")
        return []

    c = {k: (v or 0) for k, v in row.items()}
    findings = []

    # 1875-05-20 (Cobalt empty-date) -> KNOWN_NULL_SENTINEL for all column types
    if c["c_cobalt"]:
        findings.append(_finding(tbl, col, dtype, sem, "KNOWN_NULL_SENTINEL",
            c["c_cobalt"], total, ["1875-05-20"],
            "Cobalt empty-date sentinel. Recommend NULL after steward confirmation.",
            hint="A",
            auto_decided_action="convert_to_null",
        ))

    # 1900-01-01 handling depends on semantic type
    if c["c_epoch"]:
        if sem_type == "date_of_birth":
            findings.append(_finding(tbl, col, dtype, sem, "DOB_SUSPECT_SENTINEL",
                c["c_epoch"], total, ["1900-01-01"],
                "1900-01-01 in a DOB column is an unregistered sentinel. "
                "Quarantine candidate -- do NOT auto-null. Steward investigation required.",
                auto_decided_action="quarantine_pending_confirmation",
            ))
        else:
            findings.append(_finding(tbl, col, dtype, sem, "SUSPECT_SENTINEL",
                c["c_epoch"], total, ["1900-01-01"],
                "Common DB default date. Profile before action -- never auto-convert to NULL.",
                confidence="medium"
            ))

    # 9999-12-31 handling depends on semantic type
    if c["c_max"] and sem_type != "date_of_birth":
        if sem_type == "date_of_death":
            findings.append(_finding(tbl, col, dtype, sem, "DOD_SENTINEL_VIOLATION",
                c["c_max"], total, ["9999-12-31"],
                "POLICY VIOLATION (§6,§8): Living patients must have NULL DOD -- never a sentinel. "
                "9999-12-31 in DOD BLOCKS certification. Immediate governance review required.",
                auto_decided_action="convert_to_null",
            ))
        elif sem_type == "open_ended_end_date":
            findings.append(_finding(tbl, col, dtype, sem, "MAX_DATE_SENTINEL",
                c["c_max"], total, ["9999-12-31"],
                "Per MOSAIC standard, replace 9999-12-31 with 2199-01-01 for open-ended ranges.",
                hint="B",
                std_options=[o for o in STANDARDIZATION_RULES["MAX_DATE_SENTINEL"]
                             if o["option_key"] == "replace_with_mosaic_sentinel"],
                auto_decided_action="replace_with_mosaic_sentinel",
            ))
        else:
            findings.append(_finding(tbl, col, dtype, sem, "MAX_DATE_SENTINEL",
                c["c_max"], total, ["9999-12-31"],
                "9999-12-31 detected. Confirm: open-ended range (-> 2199-01-01) or point-in-time (-> NULL)?"
            ))

    # 2199-01-01 handling depends on semantic type (§3, §6)
    if c["c_mosaic"]:
        if sem_type == "date_of_death":
            # Living patients must be NULL in DOD -- never a sentinel (§6)
            findings.append(_finding(tbl, col, dtype, sem, "DOD_SENTINEL_VIOLATION",
                c["c_mosaic"], total, ["2199-01-01"],
                "POLICY VIOLATION (§6,§8): 2199-01-01 in a DOD column -- living patients must be NULL. "
                "BLOCKS certification.",
                auto_decided_action="convert_to_null",
            ))
        elif sem_type != "open_ended_end_date":
            # 2199-01-01 is only approved for open-ended end-date columns (§3).
            # In any other column type it is unexpected and should be flagged.
            findings.append(_finding(tbl, col, dtype, sem, "SUSPECT_SENTINEL",
                c["c_mosaic"], total, ["2199-01-01"],
                "2199-01-01 is the MOSAIC approved sentinel only for open-ended end-date columns "
                "(effective_to, valid_to). Its presence in this column type is unexpected -- "
                "confirm with steward whether it was applied correctly.",
                confidence="medium"
            ))

    # Invalid / zero dates
    if c["c_invalid"]:
        findings.append(_finding(tbl, col, dtype, sem, "INVALID_DATE",
            c["c_invalid"], total, [],
            "Zero or unparseable dates detected. Investigate source system. Recommend quarantine.",
            auto_decided_action="null_and_quarantine",
        ))

    # Future dates -- each semantic type gets its own branch for clarity
    # and so auto_decided_action can be set correctly per type.
    if c["c_future"]:
        if sem_type == "date_of_birth":
            findings.append(_finding(tbl, col, dtype, sem, "FUTURE_DOB",
                c["c_future"], total, [],
                "Future DOB detected. Quarantine for steward review -- never silently null (§6-DOB).", auto_decided_action="quarantine",
            ))
        elif sem_type == "date_of_death":
            findings.append(_finding(tbl, col, dtype, sem, "DOD_FUTURE",
                c["c_future"], total, [],
                "Future DOD detected. Quarantine for steward review (§6-DOD).", auto_decided_action="quarantine",
            ))
        else:
            findings.append(_finding(tbl, col, dtype, sem, "INVALID_DATE",
                c["c_future"], total, [],
                f"Future date in a {sem_type or 'date'} column. Quarantine for steward review.", auto_decided_action="null_and_quarantine",
            ))

    # §6-DOB plausibility (age < 0 or > age_max)
    if sem_type == "date_of_birth":
        try:
            p = spark.sql(f"""
                SELECT
                    COUNTIF(DATEDIFF(year, TRY_CAST({dc} AS DATE), DATE'{TODAY}') < 0)              AS neg,
                    COUNTIF(DATEDIFF(year, TRY_CAST({dc} AS DATE), DATE'{TODAY}') > {CFG['age_max']}) AS old
                FROM {fq}
                WHERE `{col}` IS NOT NULL AND TRY_CAST({dc} AS DATE) IS NOT NULL
            """).collect()[0]
            plaus = (p["neg"] or 0) + (p["old"] or 0)
            if plaus:
                findings.append(_finding(tbl, col, dtype, sem, "PLAUSIBILITY_REVIEW",
                    plaus, total, [],
                    f"DOB implies age < 0 or > {CFG['age_max']} years. Plausibility review required.",
                    auto_decided_action="quarantine",
                ))
        except Exception as e:
            print(f"  [WARN] DOB plausibility failed for {tbl}.{col}: {e}")


    # §6-DOD: DOD before DOB cross-column check
    if sem_type == "date_of_death" and dob_col:
        dob_dtype = dict(table_cols[tbl]).get(dob_col, "")
        dob_dc    = f"`{dob_col}`" if _is_dt(dob_dtype) else f"TRY_CAST(`{dob_col}` AS DATE)"
        try:
            cross = spark.sql(f"""
                SELECT COUNT(*) AS n FROM {fq}
                WHERE TRY_CAST({dc} AS DATE) IS NOT NULL
                  AND TRY_CAST({dob_dc} AS DATE) IS NOT NULL
                  AND TRY_CAST({dc} AS DATE) < TRY_CAST({dob_dc} AS DATE)
            """).collect()[0]["n"]
            if cross:
                findings.append(_finding(tbl, col, dtype, sem, "DOD_BEFORE_DOB",
                    cross, total, [f"vs. {dob_col}"],
                    f"DOD < DOB in {cross:,} row(s). Chronologically impossible. Quarantine.",
                    auto_decided_action="quarantine",
                ))
        except Exception as e:
            print(f"  [WARN] DOD<DOB check failed for {tbl}.{col}: {e}")

    # §3 Magic date: any single date appearing in > magic_pct% of non-null rows
    non_null = total - null_cnt
    if non_null > 0:
        threshold = max(1, int(non_null * CFG["magic_pct"] / 100))
        try:
            for mr in spark.sql(f"""
                SELECT TRY_CAST({dc} AS DATE) AS dv, COUNT(*) AS cnt
                FROM {fq} WHERE `{col}` IS NOT NULL
                GROUP BY 1 HAVING COUNT(*) >= {threshold}
                ORDER BY cnt DESC LIMIT 20
            """).collect():
                if mr["dv"] and mr["dv"] not in KNOWN_DATES:
                    findings.append(_finding(tbl, col, dtype, sem, "CANDIDATE_MAGIC_DATE",
                        mr["cnt"], total, [str(mr["dv"])],
                        f"Date {mr['dv']} appears in >= {CFG['magic_pct']}% of non-null rows. "
                        "Possible system default. Steward investigation required.",
                        confidence="medium"
                    ))
        except Exception as e:
            print(f"  [WARN] Magic date check failed for {tbl}.{col}: {e}")

    return findings


# ── §3/§5 Numeric sentinel rules ─────────────────────────────────────────────
def _check_numeric(tbl, col, stats, sem) -> list:
    s        = stats[col]
    dtype    = s["dtype"]
    total    = s["total"]
    sem_type = sem.get("type", "other")

    if not _is_num(dtype):
        return []

    fq = f"`{cat}`.`{sch}`.`{tbl}`"
    magic_exprs = ", ".join(
        f"COUNTIF(`{col}` = {m}) AS c_{str(m).replace('-','neg')}"
        for m in MAGIC_NUMBERS
    )
    try:
        row = spark.sql(
            f"SELECT COUNTIF(`{col}` = 0) AS c_zero, {magic_exprs} FROM {fq}"
        ).collect()[0].asDict()
    except Exception as e:
        print(f"  [WARN] Numeric check failed for {tbl}.{col}: {e}")
        return []

    findings  = []
    threshold = max(1, int(total * CFG["magic_pct"] / 100))

    if row["c_zero"] and sem_type == "measure_amount":
        findings.append(_finding(tbl, col, dtype, sem, "ZERO_AS_MISSING_REVIEW",
            row["c_zero"], total, [0],
            "Zero in a measure/amount column may represent missing data. "
            "Confirm with domain steward whether 0 is a valid business value.",
            hint="A", confidence="medium"
        ))

    for m in MAGIC_NUMBERS:
        key = f"c_{str(m).replace('-','neg')}"
        cnt = row.get(key, 0) or 0
        if cnt >= threshold:
            findings.append(_finding(tbl, col, dtype, sem, "CANDIDATE_MAGIC_NUMBER",
                cnt, total, [m],
                f"Value {m} appears in >= {CFG['magic_pct']}% of rows. "
                "Possible undocumented sentinel. Steward investigation required.",
                confidence="medium"
            ))

    return findings


# ── §2-AI: AI token classification for unseen sentinels and ambiguous codes ───
def _check_string_tokens_ai(tbl, col, stats, sem, dist_vals: list, findings_so_far: list) -> list:
    """
    Runs AFTER all deterministic checks on low-cardinality string columns.
    Classifies tokens that deterministic rules did not flag -- covering:
      - Multilingual sentinels (Desconocido, Inconnu, Unbekannt, NSP...)
      - Domain-specific codes (NI, OTH, UNKN, TBD, PEND, VOID, 9, 99...)
      - Client-specific missing markers not in the fixed allowlists
    One ai_query() call per column. Only fires when:
      1. enable_ai = True
      2. Column has <= 30 distinct values (low-cardinality -- code/sentinel territory)
      3. At least one token was not claimed by any deterministic rule
    """
    if not CFG["enable_ai"] or not CFG["enable_ai_token_discovery"]:
        return []

    sem_type = sem.get("type", "other")
    if sem_type in ("person_name", "identifier", "free_text", "date_of_birth", "date_of_death"):
        return []
    n_distinct = int(stats[col].get("approx_distinct", len(dist_vals)) or 0)

    # Only useful on low-cardinality columns
    if n_distinct > 30 or n_distinct == 0:
        return []

    # Collect tokens already claimed by deterministic findings
    claimed_samples = set()
    for f in findings_so_far:
        try:
            svs = json.loads(f.get("sample_values", "[]"))
            for sv in svs:
                claimed_samples.add(str(sv).strip().lower())
        except Exception:
            pass

    # Also exclude tokens in the fixed allowlists (already handled deterministically)
    known = NULL_TOKENS | SENTINEL_TOKENS | {"y","n","yes","no","true","false","1","0","t","f"}

    unclaimed = [
        (tv, cnt) for tv, cnt in dist_vals
        if tv.strip() != ""
        and tv.strip().lower() not in known
        and tv.strip().lower() not in claimed_samples
    ]
    if not unclaimed:
        return []

    total = stats[col]["total"]
    dtype = stats[col]["dtype"]

    payload = json.dumps(
        [{"token": tv, "count": cnt, "pct": round(cnt / max(sum(c for _, c in dist_vals), 1) * 100, 1)}
         for tv, cnt in unclaimed[:25]],
        default=str
    )
    prompt = (
        "Data quality classifier. Reply ONLY with a JSON array -- no prose, no markdown. "
        f"Table: {tbl}, Column: {col} (semantic type: {sem_type}, data type: {dtype}). "
        f"Total rows: {total}. Distinct values in column: {n_distinct}. "
        "Classify each token as exactly one of: "
        "missing_marker (a placeholder meaning data was not collected, unknown, or not applicable -- "
        "includes multilingual variants like Desconocido, Inconnu, Unbekannt, NSP, NI, 9, 99, UNKN, TBD, PEND, VOID, NO DATA, NOT_KNOWN), "
        "business_sentinel (a coded value with domain meaning that should be preserved -- e.g. N/A, Unknown, None), "
        "valid_code (a genuine business value -- e.g. M/F for gender, status codes), "
        "other (cannot determine from context alone). "
        f"Tokens: {payload}. "
        'Return: [{"token":"<t>","classification":"<c>","confidence":"high|medium|low","rationale":"<1 sentence>"}]'
    )

    try:
        parsed = json.loads(_ai_query(prompt))
    except Exception as e:
        print(f"  [WARN] AI token classification failed for {tbl}.{col}: {e}")
        return []

    tok_map = {tv: cnt for tv, cnt in unclaimed}
    findings = []

    for item in parsed:
        tv  = item.get("token", "")
        cls = item.get("classification", "other")
        conf = item.get("confidence", "low")
        rationale = item.get("rationale", "")
        cnt = tok_map.get(tv, 0)

        if cls == "missing_marker":
            exact_cnt = spark.table(_fq(cat, sch, tbl)).filter(F.trim(F.col(_qident(col))) == F.lit(tv)).count()
            findings.append(_finding(tbl, col, dtype, sem, "UNREGISTERED_MISSING_MARKER",
                exact_cnt, total, mask([tv]),
                f"AI identified '{tv}' as a potential missing-value marker "
                f"({rationale}). Steward must confirm before any transform.",
                confidence=conf,
                std_options=[
                    {"option_key": "preserve_pending_review", "label": "Preserve pending steward review", "sql": "col", "notes": "Default safe action until the marker is registered."},
                    {"option_key": "convert_to_null_confirmed", "label": "Convert confirmed marker to SQL NULL", "sql": f"CASE WHEN TRIM(col) = {_sql_lit(tv)} THEN NULL ELSE TRIM(col) END", "notes": "Requires source-specific profiling, dictionary registration, steward sign-off, and a version bump."},
                ],
            ))
        elif cls == "business_sentinel":
            findings.append(_finding(tbl, col, dtype, sem, "PRESERVE_SENTINEL",
                cnt, total, mask([tv]),
                f"AI identified '{tv}' as a business sentinel that should be preserved "
                f"({rationale}). Normalise to canonical form; do NOT convert to NULL.",
                hint="B", confidence=conf,
                auto_decided_action="normalise_canonical",
            ))
        # valid_code and other are not flagged -- they are not missing-value patterns

    return findings


# ── §1-AI: AI strategy hint for STRING NULL_PROFILE ──────────────────────────
def _check_null_profile_strategy_ai(tbl, col, stats, sem, null_profile_fired: bool) -> list:
    """
    When a string column fires NULL_PROFILE with no other specific findings,
    ask the AI which data-dictionary strategy (A/B/C/D) best fits based on
    column name, table, semantic type, and null rate.
    Returns an updated NULL_PROFILE finding with a strategy hint, or [].
    One ai_query() call per qualifying column.
    """
    if not CFG["enable_ai"] or not null_profile_fired:
        return []
    if not _is_str(stats[col]["dtype"]):
        return []

    s        = stats[col]
    total    = s["total"]
    null_pct = round(s["null"] / total * 100, 1) if total else 0
    sem_type = sem.get("type", "other")

    prompt = (
        "Data governance advisor. Reply ONLY with a JSON object -- no prose, no markdown. "
        f"Table: {tbl}, Column: {col}, Semantic type: {sem_type}. "
        f"This STRING column has {null_pct}% NULL values. "
        "Recommend which MOSAIC data-dictionary strategy best fits: "
        "A=data was collected but is absent (nullable, convert blank to NULL); "
        "B=data was never captured for this record by design (use a curated sentinel); "
        "C=value is required, zero nulls tolerated, quarantine missing rows; "
        "D=field does not apply to this record type. "
        "Consider the column name, table name, and semantic type carefully. "
        "Examples: a required patient identifier -> C; an optional notes field -> A; "
        "a field that only applies to inpatients -> D for outpatient records. "
        'Return: {"strategy":"A|B|C|D","confidence":"high|medium|low","rationale":"<1 sentence>"}'
    )

    try:
        result = json.loads(_ai_query(prompt))
        strategy = result.get("strategy", "").upper()
        if strategy not in ("A", "B", "C", "D"):
            return []
        conf      = result.get("confidence", "low")
        rationale = result.get("rationale", "")
        return [(strategy, conf, rationale)]   # caller applies this to the NULL_PROFILE finding
    except Exception as e:
        print(f"  [WARN] AI strategy hint failed for {tbl}.{col}: {e}")
        return []


# ── Main loop ────────────────────────────────────────────────────────────────
all_findings: List[dict] = []

for tbl, stats in table_stats.items():
    sem_tbl   = sem_map.get(tbl, {})
    sample_df = table_samples[tbl]
    cols      = table_cols[tbl]

    # Identify DOB column for DOD cross-check
    dob_col = next(
        (c for c, _ in cols if sem_tbl.get(c, {}).get("type") == "date_of_birth"),
        None
    )

    tbl_count = 0
    for col, dtype in cols:
        sem = sem_tbl.get(col, {"type": "other", "source": "heuristic", "confidence": "heuristic", "needs_review": False})
        # Low-cardinality distributions are counted against the full table so
        # affected_count is exact. High-cardinality columns use a bounded discovery sample;
        # fixed-token counts still come from the exact aggregate profile.
        dist_vals_main = None
        if _is_str(stats[col]["dtype"]):
            distribution_df = (
                spark.table(_fq(cat, sch, tbl))
                if int(stats[col].get("approx_distinct", 0) or 0) <= CFG["token_limit"]
                else sample_df
            )
            dist_vals_main = [(r["tv"], r["count"]) for r in (
                distribution_df
                .select(F.trim(F.col(_qident(col))).alias("tv"))
                .filter(F.col("tv").isNotNull())
                .groupBy("tv").count()
                .orderBy(F.col("count").desc())
                .limit(CFG["token_limit"])
                .collect()
            )]

        # Step 1: run all deterministic checks
        deterministic = (
            _check_null_profile(tbl, col, stats, sem)
            + _check_string_tokens(tbl, col, stats, sem, sample_df,
                                   dist_vals_in=dist_vals_main)
            + _check_date_sentinels(tbl, col, stats, sem, dob_col)
            + _check_numeric(tbl, col, stats, sem)
        )

        # Step 2: AI token classification for unseen sentinels/codes.
        # Only runs on low-cardinality string columns after deterministic checks.
        # Catches multilingual sentinels, domain codes, and client-specific markers
        # that fixed allowlists (SENTINEL_TOKENS, AMBIG_CODES) would miss.
        ai_token_findings = []
        if _is_str(stats[col]["dtype"]) and dist_vals_main:
            ai_token_findings = _check_string_tokens_ai(
                tbl, col, stats, sem, dist_vals_main, deterministic
            )

        new = deterministic + ai_token_findings

        # Suppress NULL_PROFILE when more specific findings already cover this column.
        tags_fired = {f["classification"] for f in new}
        if "NULL_PROFILE" in tags_fired and len(tags_fired) > 1:
            new = [f for f in new if f["classification"] != "NULL_PROFILE"]

        all_findings.extend(new)
        tbl_count += len(new)

    print(f"  ok {tbl}: {tbl_count} finding(s)")

print(f"\nRule engine done. {len(all_findings)} finding(s).")


In [0]:
# The only write in this notebook. Source tables are never modified.

from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, BooleanType

SCHEMA = StructType([
    StructField("run_id",                   StringType(),  True),
    StructField("run_ts",                   StringType(),  True),
    StructField("catalog",                  StringType(),  True),
    StructField("schema",                   StringType(),  True),
    StructField("table_name",               StringType(),  True),
    StructField("column_name",              StringType(),  True),
    StructField("data_type",                StringType(),  True),
    StructField("semantic_type",            StringType(),  True),
    StructField("semantic_source",          StringType(),  True),
    StructField("rule_ref",                 StringType(),  True),
    StructField("classification",           StringType(),  True),
    StructField("affected_count",           LongType(),    True),
    StructField("affected_pct",             DoubleType(),  True),
    StructField("total_rows",               LongType(),    True),
    StructField("sample_values",            StringType(),  True),
    StructField("recommended_action",       StringType(),  True),
    StructField("dictionary_strategy_hint", StringType(),  True),
    StructField("confidence",               StringType(),  True),
    StructField("needs_steward_review",     BooleanType(), True),
    StructField("standardization_rule",     StringType(),  True),
    # Decision columns -- policy engine for guarded single-option rules; otherwise steward workflow.
    # decided_action : the option_key from standardization_rule the steward selected.
    # decided_by     : free-text name or email of the steward who made the decision.
    StructField("decided_action",            StringType(),  True),
    StructField("decided_by",                StringType(),  True),
])

out_fq = _fq(CFG["out_catalog"], CFG["out_schema"], CFG["out_table"])

findings_df = spark.createDataFrame(all_findings or [], schema=SCHEMA)

if all_findings:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {_fq(CFG['out_catalog'], CFG['out_schema'])}")
    findings_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(out_fq)
    print(f"ok  {len(all_findings):,} finding(s) written to {out_fq}")
else:
    print("No findings generated.")

print(f"Run ID: {RUN_ID}")


In [0]:
# ---------------------------------------------------------------------------
# Section 1 -- Blocking findings (read first -- these block certification)
# Section 2 -- Findings by classification
# Section 3 -- Findings by table
# Section 4 -- Top-20 highest-impact columns (affected_pct x severity)
# ---------------------------------------------------------------------------

BLOCKING = {
    "DOD_SENTINEL_VIOLATION", "FUTURE_DOB", "DOD_FUTURE",
    "DOD_BEFORE_DOB", "DOB_SUSPECT_SENTINEL"
}

if not all_findings:
    print("No findings. Run complete.")
else:
    fdf = findings_df

    # Section 1 -- Blocking
    block_df = (
        fdf.filter(F.col("classification").isin(BLOCKING))
           .orderBy("classification", F.col("affected_pct").desc())
    )
    n_block = block_df.count()
    print("=" * 68)
    print(f"  BLOCKING FINDINGS: {n_block}  (block certification per §8)")
    print("=" * 68)
    if n_block:
        display(block_df.select(
            "table_name","column_name","classification","rule_ref",
            "affected_count","affected_pct","recommended_action",
            "standardization_rule","decided_action","decided_by"
        ))
    else:
        print("  None detected.")

    # Section 2 -- By classification
    print("\n" + "-" * 68)
    print("SECTION 2 -- Findings by classification")
    print("-" * 68)
    display(
        fdf.groupBy("classification","rule_ref")
           .agg(
               F.count("*").alias("findings"),
               F.sum("affected_count").alias("total_affected_rows"),
               F.countDistinct("table_name").alias("tables"),
               F.countDistinct("column_name").alias("columns"),
           ).orderBy(F.col("findings").desc())
    )

    # Section 3 -- By table
    print("\n" + "-" * 68)
    print("SECTION 3 -- Findings by table")
    print("-" * 68)
    display(
        fdf.groupBy("table_name")
           .agg(
               F.count("*").alias("total_findings"),
               F.sum(F.when(F.col("classification").isin(BLOCKING), 1).otherwise(0)).alias("blocking"),
               F.sum(F.when(F.col("needs_steward_review"), 1).otherwise(0)).alias("steward_reviews"),
           ).orderBy(F.col("blocking").desc(), F.col("total_findings").desc())
    )

    # Section 4 -- Top-20 by impact score
    print("\n" + "-" * 68)
    print("SECTION 4 -- Top-20 highest-impact columns  (affected_pct x severity)")
    print("-" * 68)
    sev_expr = F.create_map(*[
        item for k, v in SEVERITY.items() for item in (F.lit(k), F.lit(v))
    ])
    display(
        fdf
        .withColumn("severity",     sev_expr[F.col("classification")])
        .withColumn("impact_score", F.round(F.col("affected_pct") * F.coalesce(F.col("severity"), F.lit(1)), 4))
        .orderBy(F.col("impact_score").desc())
        .limit(20)
        .select("table_name","column_name","classification","rule_ref",
                "affected_pct","severity","impact_score","confidence",
                "recommended_action","standardization_rule")
    )

    # ── Section 5 -- Engineer work queue (decisions recorded) ────────────
    print("\n" + "-" * 68)
    print("SECTION 5 -- Engineer work queue  (decided_action IS NOT NULL)")
    print("-" * 68)
    work_df = fdf.filter(F.col("decided_action").isNotNull())
    n_work  = work_df.count()
    if n_work:
        display(work_df.select(
            "table_name","column_name","classification","decided_action",
            "decided_by","standardization_rule"
        ).orderBy("table_name","column_name"))
    else:
        print("  No decisions recorded yet.")
        print("  To record a decision: UPDATE the findings table and set")
        print("  decided_action = <option_key> and decided_by = <your name>")
        print("  for the relevant (run_id, table_name, column_name, classification) row.")

    print("\n" + "=" * 68)
    print(f"  Run: {RUN_ID}")
    print(f"  Scope: {cat}.{sch}  ({len(table_stats)} table(s))")
    print(f"  AI: {'on' if CFG['enable_ai'] else 'off'}  |  Findings: {len(all_findings):,}  |  Blocking: {n_block}")
    print(f"  Results: {out_fq}")
    print("=" * 68)
    print()
    print("Detection-only. No source data was modified.")
    print("Every finding requires data steward review before any action.")
